# 🚀 CLIR-SkateFormer: All-in-One Google Colab & Local Notebook (NTU60 Cross-Subject / 60CSub)
> **Mô hình**: **CLIR-SkateFormer (Concept Logic Iterative Refinement for Skeleton Action Recognition)**  
> **Benchmark mặc định**: **NTU RGB+D 60 Cross-Subject (60CSub / NTU60_CS)**  
> **Số lớp hành động**: **60 classes** (A1–A60)  
> **Số khái niệm**: **72 concepts** (52 spatial + 20 temporal)  
> **Điểm đột phá**:
> 1. Mã hóa tri thức Action-Concept trực tiếp bằng luật DNF phân cấp (L1 part-grouped AND → L2a AND2 ∥ L2b OR1 → Strict ActionHead).
> 2. Loại bỏ hoàn toàn ma trận tương quan Concept-Concept tĩnh $72 \times 72$.
> 3. Tinh chỉnh khái niệm vòng lặp ($K=2$) dẫn dắt bởi **Logic Evidence** mức cao ($r^t \in \mathbb{R}^{512}$).
> 4. Huấn luyện 100% Pure STE (Straight-Through Estimator) với Continuous Temperature Annealing ($T_{\text{start}}=2.0 \rightarrow T_{\text{end}}=10.0$).

---

### 📌 Cấu trúc Toàn diện Notebook:
1. **Block 1**: Môi trường, Dependencies & Mount Google Drive
2. **Block 2**: Concept Ontology CSV (60/120 actions $\times$ 72 concepts) & NTU Feeder
3. **Block 3**: Perception Backbone SkateFormer
4. **Block 4**: Tầng Logic CRL (LiteralLayer, ConjunctionLayer, DisjunctionLayer, CRLReasoner)
5. **Block 5**: ConceptUpdateCell, IterativeLogicReasoner & CLIRSkateFormer
6. **Block 6**: RuleExtractor (Post-hoc DNF Rule Extraction & Fidelity Engine)
7. **Block 7**: Kiểm thử nhanh tính toàn vẹn (Quick Smoke Test cho 60CSub)
8. **Block 8**: Thiết lập Cấu hình & DataLoaders (Mặc định NTU60 Cross-Subject)
9. **Block 9**: Vòng lặp Huấn luyện 100% Pure STE (hoặc 2-Phase fallback) & Lưu Checkpoint
10. **Block 10**: Trích xuất & Trực quan hóa Luật DNF (Precision, Coverage, Fidelity)
11. **Block 11**: Huấn luyện qua dòng lệnh CLI (Alternative CLI Execution cho NTU60 CS)



## 📦 Block 1: Cài đặt Dependencies & Môi trường


In [1]:
import os
import sys
import time
import json
import torch
import numpy as np

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")



PyTorch Version: 2.11.0+cu128
CUDA Available: True
Device: NVIDIA GeForce RTX 5090


## 📑 Block 2: Concept Ontology & Data Feeder


In [2]:
# Lưu concept ontology CSV vào workspace (72 concepts chuẩn hóa)
os.makedirs("concepts", exist_ok=True)
with open("concepts/ntu_rgbd_60_120.csv", "w", encoding="utf-8") as f:
    f.write('''action_class,head_nod,head_shake,head_tilt_down,head_tilt_up,head_rotate,head_static,hand_grasp,hand_release,hand_wave,hand_point,hand_clap,hand_manipulate,hand_fist,hand_open,hand_touch,hand_lift_to_face,hand_circular,arm_raise,arm_lower,arm_extend,arm_retract,arm_circular,arm_swing,arm_cross,arm_hold,arm_alternating,arm_overhead,arm_lateral,hip_bend,hip_extend,hip_sit,hip_squat,hip_rotate,hip_static_stand,hip_static_sit,hip_sway,leg_step,leg_kick,leg_jump,leg_squat,leg_lift,leg_static_stand,leg_static_sit,leg_alternating,leg_balance,foot_step,foot_kick,foot_jump,foot_pivot,foot_static,foot_slide,interaction_two_person,temporal_direction_motion_upward,temporal_direction_motion_downward,temporal_direction_motion_forward,temporal_direction_motion_backward,temporal_direction_motion_converging,temporal_direction_motion_diverging,temporal_direction_motion_reversible,temporal_sequence_seq_hands_first,temporal_sequence_seq_legs_first,temporal_sequence_seq_body_first,temporal_sequence_seq_simultaneous,temporal_sequence_seq_alternating,temporal_dynamics_speed_slow,temporal_dynamics_speed_fast,temporal_dynamics_speed_accelerating,temporal_dynamics_speed_decelerating,temporal_dynamics_rhythm_regular,temporal_dynamics_rhythm_irregular,temporal_dynamics_duration_brief,temporal_dynamics_duration_sustained
drink_water,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0
eat_meal,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
brush_teeth,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1
brush_hair,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
drop,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0
pick_up,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0
throw,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
sit_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0
stand_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0
clapping,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,1,0,0,1,0,0,1
reading,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
writing,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1
tear_up_paper,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0
put_on_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1
take_off_jacket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1
put_on_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0
take_off_a_shoe,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,0,0
put_on_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_glasses,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
put_on_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_a_hat_cap,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
cheer_up,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
hand_waving,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
kicking_something,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
reach_into_pocket,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
hopping,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1
jump_up,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
phone_call,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
play_with_phone_tablet,0,0,1,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1
type_on_a_keyboard,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,1
point_to_something,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
taking_a_selfie,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
check_time_(from_watch),0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
rub_two_hands,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1
nod_head_bow,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0
shake_head,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0
wipe_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0
salute,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
put_palms_together,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
cross_hands_in_front,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
sneeze_cough,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
staggering,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1
falling_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
headache,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
chest_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
back_pain,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
neck_pain,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1
nausea_vomiting,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0
fan_self,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,1
punch_slap,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
kicking,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
pushing,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0
pat_on_back,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
point_finger,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
hugging,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
giving_object,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
touch_pocket,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
shaking_hands,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0
walking_towards,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
walking_apart,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
put_on_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,1,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_headphone,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,1,0
shoot_at_basket,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
bounce_ball,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1
tennis_bat_swing,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
juggle_table_tennis_ball,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1
hush,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
flick_hair,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
thumb_up,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
thumb_down,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
make_OK_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
make_victory_sign,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
staple_book,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
counting_money,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
cutting_nails,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,1
cutting_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
snap_fingers,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
open_bottle,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1,0
sniff_smell,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,0
squat_down,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0
toss_a_coin,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
fold_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
ball_up_paper,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0
play_magic_cube,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,1
apply_cream_on_face,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
apply_cream_on_hand,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,0,1
put_on_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
take_off_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0
put_object_into_bag,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
take_object_out_of_bag,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
open_a_box,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,1,0,0,0,0,0,1,0
move_heavy_objects,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1
shake_fist,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,1,0
throw_up_cap_hat,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
capitulate,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0
cross_arms,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1
arm_circles,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,1
arm_swings,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,1,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,1
run_on_the_spot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1
butt_kicks,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,1
cross_toe_touch,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,1
side_kick,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
yawn,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1
stretch_oneself,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1
blow_nose,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
hit_with_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0
wield_knife,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
knock_over,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
grab_stuff,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,1,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0
shoot_with_gun,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
step_on_foot,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,1,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,1,0
high-five,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0
cheers_and_drink,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0
carry_object,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
take_a_photo,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0
follow,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1
whisper,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
exchange_things,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0
support_somebody,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
rock-paper-scissors,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,1,1,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,1,0,1,0
''')
print("Concept CSV written: concepts/ntu_rgbd_60_120.csv (72 concepts)")



Concept CSV written: concepts/ntu_rgbd_60_120.csv (72 concepts)


In [3]:
# Data Augmentation Tools
import os
import sys
import types

# 1. Lưu feeders/tools.py vào workspace để DataLoader workers có thể import feeders.tools bình thường
os.makedirs("feeders", exist_ok=True)
with open("feeders/__init__.py", "w", encoding="utf-8") as f:
    f.write("")
with open("feeders/tools.py", "w", encoding="utf-8") as f:
    f.write("import random\nimport numpy as np\nimport math\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\ndef valid_crop_resize(data_numpy, valid_frame_num, p_interval, window, thres):\n    # input: C,T,V,M\n    if isinstance(p_interval, (int, float)):\n        p_interval = [p_interval]\n    C, T, V, M = data_numpy.shape\n    begin = 0\n    end = valid_frame_num if valid_frame_num > 0 else T\n    valid_size = max(1, end - begin)\n\n    # crop\n    if len(p_interval) == 1:\n        p = p_interval[0]\n        bias = int((1 - p) * valid_size / 2)\n        c_b = min(begin + bias, T - 1)\n        c_e = max(c_b + 1, min(end - bias, T))\n        data = data_numpy[:, c_b:c_e, :, :]  # center_crop\n        cropped_length = data.shape[1]\n    else:\n        p = float(np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0])\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)  # constraint cropped_length lower bound as thres\n        max_bias = max(0, valid_size - cropped_length)\n        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0\n        c_b = begin + bias\n        c_e = min(begin + bias + cropped_length, T)\n        data = data_numpy[:, c_b:c_e, :, :]\n        if data.shape[1] == 0:\n            print(cropped_length, bias, valid_size)\n\n    # resize\n    data = torch.tensor(data, dtype=torch.float)  # C, crop_t, V, M\n    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, cropped_length)  # V*M, C, crop_t\n    data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T\n    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()\n    index_t = torch.arange(start=c_b, end=c_e, dtype=torch.float)\n    index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()\n    index_t = 2 * index_t / valid_size - 1\n    return data, index_t.numpy()\n\n\ndef valid_crop_uniform(data_numpy, valid_frame_num, p_interval, window, thres):\n    # input: C,T,V,M\n    if isinstance(p_interval, (int, float)):\n        p_interval = [p_interval]\n    C, T, V, M = data_numpy.shape\n    begin = 0\n    end = valid_frame_num if valid_frame_num > 0 else T\n    valid_size = max(1, end - begin)\n\n    # crop\n    if len(p_interval) == 1:\n        p = p_interval[0]\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)\n        bias = int((1 - p) * valid_size / 2)\n        bias = min(bias, max(0, valid_size - cropped_length))\n\n        if cropped_length < window:\n            inds = np.arange(cropped_length)\n        else:\n            bids = np.array(\n                [i * cropped_length // window for i in range(window + 1)])\n            bst = bids[:window]\n            inds = bst\n\n        inds = inds + bias\n        inds = np.clip(inds, 0, T - 1)\n        data = data_numpy[:, inds, :, :]\n\n    else:\n        p = float(np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0])\n        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres),\n                                    valid_size)  # constraint cropped_length lower bound as 64\n        max_bias = max(0, valid_size - cropped_length)\n        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0\n\n        if cropped_length < window:\n            inds = np.arange(cropped_length)\n        elif window <= cropped_length < 2 * window:\n            basic = np.arange(window)\n            inds = np.random.choice(window + 1, cropped_length - window, replace=False)\n            offset = np.zeros(window + 1, dtype=np.int64)\n            offset[inds] = 1\n            offset = np.cumsum(offset)\n            inds = basic + offset[:-1]\n        else:\n            bids = np.array([i * cropped_length // window for i in range(window + 1)])\n            bsize = np.diff(bids)\n            bst = bids[:window]\n            offset = np.random.randint(bsize)\n            inds = bst + offset\n\n        inds = inds + bias\n        inds = np.clip(inds, 0, T - 1)\n        data = data_numpy[:, inds, :, :]\n        if data.shape[1] == 0:\n            print(cropped_length, bias, valid_size)\n\n    # resize\n    data = torch.tensor(data, dtype=torch.float)\n    index_t = torch.tensor(inds, dtype=torch.float)\n    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, len(inds))  # V*M, C, crop_t\n\n    if len(inds) != window:\n        data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T\n        index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()\n\n    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()\n    index_t = 2 * index_t / valid_size - 1\n    return data, index_t.numpy()\n\n\ndef scale(data_numpy, scale=0.2, p=0.5):\n    if random.random() < p:\n        scale = 1 + np.random.uniform(-1, 1, size=(3, 1, 1, 1)) * np.array(scale)\n        return data_numpy * scale\n    else:\n        return data_numpy.copy()\n\n\n''' AimCLR '''\ntransform_order = {'ntu': [0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7, 16, 17, 18, 19, 12, 13, 14, 15, 20, 23, 24, 21, 22]}\n\n\ndef subtract(data_numpy, p=0.5):\n    joint = random.randint(0, 24)\n\n    C, T, V, M = data_numpy.shape\n    if random.random() < p:\n        data_numpy_new = np.zeros((C, T, V, M))\n        for i in range(V):\n            data_numpy_new[:, :, i, :] = data_numpy[:, :, i, :] - data_numpy[:, :, joint, :]\n        return data_numpy_new\n    else:\n        return data_numpy.copy()\n\n\ndef temporal_flip(data_numpy, index_t, p=0.5):\n    C, T, V, M = data_numpy.shape\n    if random.random() < p:\n        time_range_order = [i for i in range(T)]\n        time_range_reverse = list(reversed(time_range_order))\n        return data_numpy[:, time_range_reverse, :, :], -index_t\n    else:\n        return data_numpy.copy(), index_t.copy()\n\n\ndef spatial_flip(data_numpy, p=0.5):\n    if random.random() < p:\n        index = transform_order['ntu']\n        return data_numpy[:, :, index, :]\n    else:\n        return data_numpy.copy()\n\n\ndef rotate(data_numpy, axis=None, angle=None, p=0.5):\n    if axis != None:\n        axis_next = axis\n    else:\n        axis_next = random.randint(0, 2)\n\n    if angle != None:\n        angle_next = random.uniform(-angle, angle)\n    else:\n        angle_next = random.uniform(-30, 30)\n\n    if random.random() < p:\n        temp = data_numpy.copy()\n        angle = math.radians(angle_next)\n        # x\n        if axis_next == 0:\n            R = np.array([[1, 0, 0],\n                          [0, math.cos(angle), math.sin(angle)],\n                          [0, -math.sin(angle), math.cos(angle)]])\n        # y\n        if axis_next == 1:\n            R = np.array([[math.cos(angle), 0, -math.sin(angle)],\n                          [0, 1, 0],\n                          [math.sin(angle), 0, math.cos(angle)]])\n        # z\n        if axis_next == 2:\n            R = np.array([[math.cos(angle), math.sin(angle), 0],\n                          [-math.sin(angle), math.cos(angle), 0],\n                          [0, 0, 1]])\n        R = R.transpose()\n        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)\n        temp = temp.transpose(3, 0, 1, 2)\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef shear(data_numpy, s1=None, s2=None, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        if s1 != None:\n            s1_list = s1\n        else:\n            s1_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]\n        if s2 != None:\n            s2_list = s2\n        else:\n            s2_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]\n\n        R = np.array([[1, s1_list[0], s2_list[0]],\n                      [s1_list[1], 1, s2_list[1]],\n                      [s1_list[2], s2_list[2], 1]])\n        R = R.transpose()\n        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)\n        temp = temp.transpose(3, 0, 1, 2)\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef drop_axis(data_numpy, axis=None, p=0.5):\n    if axis != None:\n        axis_next = axis\n    else:\n        axis_next = random.randint(0, 2)\n\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n        x_new = np.zeros((T, V, M))\n        temp[axis_next] = x_new\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef drop_joint(data_numpy, joint_list=None, time_range=None, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n\n        if joint_list != None:\n            all_joints = [i for i in range(V)]\n            joint_list_ = random.sample(all_joints, joint_list)\n            joint_list_ = sorted(joint_list_)\n        else:\n            random_int = random.randint(5, 15)\n            all_joints = [i for i in range(V)]\n            joint_list_ = random.sample(all_joints, random_int)\n            joint_list_ = sorted(joint_list_)\n\n        if time_range != None:\n            all_frames = [i for i in range(T)]\n            time_range_ = random.sample(all_frames, time_range)\n            time_range_ = sorted(time_range_)\n        else:\n            random_int = random.randint(16, 32)\n            all_frames = [i for i in range(T)]\n            time_range_ = random.sample(all_frames, random_int)\n            time_range_ = sorted(time_range_)\n\n        x_new = np.zeros((C, len(time_range_), len(joint_list_), M))\n        temp2 = temp[:, time_range_, :, :].copy()\n        temp2[:, :, joint_list_, :] = x_new\n        temp[:, time_range_, :, :] = temp2\n        return temp\n    else:\n        return data_numpy.copy()\n\n\ndef gaussian_noise(data_numpy, mean=0, std=0.05, p=0.5):\n    if random.random() < p:\n        temp = data_numpy.copy()\n        C, T, V, M = data_numpy.shape\n        noise = np.random.normal(mean, std, size=(C, T, V, M))\n        return temp + noise\n    else:\n        return data_numpy.copy()\n\n\nclass GaussianBlurConv(nn.Module):\n    def __init__(self, channels=3, kernel=15, sigma=[0.1, 2], p=0.5):\n        super(GaussianBlurConv, self).__init__()\n        self.channels = channels\n        self.kernel = kernel\n        self.min_max_sigma = sigma\n        radius = int(kernel / 2)\n        self.kernel_index = np.arange(-radius, radius + 1)\n        self.p = p\n\n    def __call__(self, x):\n        sigma = random.uniform(self.min_max_sigma[0], self.min_max_sigma[1])\n        blur_flter = np.exp(-np.power(self.kernel_index, 2.0) / (2.0 * np.power(sigma, 2.0)))\n        kernel = torch.from_numpy(blur_flter).unsqueeze(0).unsqueeze(0)\n        # kernel =  kernel.float()\n        kernel = kernel.double()\n        kernel = kernel.repeat(self.channels, 1, 1, 1)\n        self.weight = nn.Parameter(data=kernel, requires_grad=False)\n\n        prob = np.random.random_sample()\n        x = torch.from_numpy(x).double()\n        if prob < self.p:\n            x = x.permute(3, 0, 2, 1)\n            x = F.conv2d(x, self.weight, padding=(0, int((self.kernel - 1) / 2)), groups=self.channels)\n            x = x.permute(1, -1, -2, 0)\n\n        return x.numpy()\n\n\ndef gaussian_filter(data_numpy, kernel=15, sig_list=[0.1, 2], p=0.5):\n    g = GaussianBlurConv(3, kernel, sig_list, p)\n    return g(data_numpy)\n\n\n''' Skeleton AdaIN '''\ndef skeleton_adain_bone_length(input, ref): # C T V M\n    eps = 1e-5\n    center = 1\n    ref_c = ref[:, :, center, :]\n\n    # joint to bone (joint2bone)\n    j2b = joint2bone()\n    bone_i = j2b(input) # C T V M\n    bone_r = j2b(ref)\n\n    bone_length_i = np.linalg.norm(bone_i, axis=0) # T V M\n    bone_length_r = np.linalg.norm(bone_r, axis=0)\n\n    bone_length_scale = (bone_length_r + eps) / (bone_length_i + eps) # T V M\n    bone_length_scale = np.expand_dims(bone_length_scale, axis=0) # 1 T V M\n\n    bone_i = bone_i * bone_length_scale\n\n    # bone to joint (bone2joint)\n    b2j = bone2joint()\n    joint = b2j(bone_i, ref_c)\n    return joint\n\n\nclass joint2bone(nn.Module):\n    def __init__(self):\n        super(joint2bone, self).__init__()\n        self.pairs = [(0, 1), (1, 1), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5), (7, 6), (8, 20), (9, 8),\n                      (10, 9), (11, 10), (12, 0), (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),\n                      (19, 18), (20, 1), (21, 7), (22, 7), (23, 11), (24, 11)]\n\n    def __call__(self, joint):\n        bone = np.zeros_like(joint)\n        for v1, v2 in self.pairs:\n            bone[:, :, v1, :] = joint[:, :, v1, :] - joint[:, :, v2, :]\n        return bone\n\n\nclass bone2joint(nn.Module):\n    def __init__(self):\n        super(bone2joint, self).__init__()\n        self.center = 1\n        self.pairs_1 = [(0, 1), (20, 1)]\n        self.pairs_2 = [(2, 20), (4, 20), (8, 20), (12, 0), (16, 0)]\n        self.pairs_3 = [(3, 2), (5, 4), (9, 8), (13, 12), (17, 16)]\n        self.pairs_4 = [(6, 5), (10, 9), (14, 13), (18, 17)]\n        self.pairs_5 = [(7, 6), (11, 10), (15, 14), (19, 18)]\n        self.pairs_6 = [(21, 7), (22, 7), (23, 11), (24, 11)]\n\n    def __call__(self, bone, center):\n        joint = np.zeros_like(bone)\n        joint[:, :, self.center, :] = center\n        for v1, v2 in self.pairs_1:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_2:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_3:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_4:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_5:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        for v1, v2 in self.pairs_6:\n            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]\n        return joint\n\ndef to_motion(input): # C T V M\n    C, T, V, M = input.shape\n    motion = np.zeros_like(input)\n    motion[:, :T - 1] = np.diff(input, axis=1)\n    return motion")

if "." not in sys.path:
    sys.path.insert(0, ".")

# 2. Định nghĩa hàm augmentation trong notebook scope
import random
import numpy as np
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def valid_crop_resize(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        bias = int((1 - p) * valid_size / 2)
        c_b = min(begin + bias, T - 1)
        c_e = max(c_b + 1, min(end - bias, T))
        data = data_numpy[:, c_b:c_e, :, :]  # center_crop
        cropped_length = data.shape[1]
    else:
        p = float(np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0])
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)  # constraint cropped_length lower bound as thres
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0
        c_b = begin + bias
        c_e = min(begin + bias + cropped_length, T)
        data = data_numpy[:, c_b:c_e, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)  # C, crop_t, V, M
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, cropped_length)  # V*M, C, crop_t
    data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = torch.arange(start=c_b, end=c_e, dtype=torch.float)
    index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def valid_crop_uniform(data_numpy, valid_frame_num, p_interval, window, thres):
    # input: C,T,V,M
    if isinstance(p_interval, (int, float)):
        p_interval = [p_interval]
    C, T, V, M = data_numpy.shape
    begin = 0
    end = valid_frame_num if valid_frame_num > 0 else T
    valid_size = max(1, end - begin)

    # crop
    if len(p_interval) == 1:
        p = p_interval[0]
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres), valid_size)
        bias = int((1 - p) * valid_size / 2)
        bias = min(bias, max(0, valid_size - cropped_length))

        if cropped_length < window:
            inds = np.arange(cropped_length)
        else:
            bids = np.array(
                [i * cropped_length // window for i in range(window + 1)])
            bst = bids[:window]
            inds = bst

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]

    else:
        p = float(np.random.rand(1) * (p_interval[1] - p_interval[0]) + p_interval[0])
        cropped_length = np.minimum(np.maximum(int(np.floor(valid_size * p)), thres),
                                    valid_size)  # constraint cropped_length lower bound as 64
        max_bias = max(0, valid_size - cropped_length)
        bias = np.random.randint(0, max_bias + 1) if max_bias > 0 else 0

        if cropped_length < window:
            inds = np.arange(cropped_length)
        elif window <= cropped_length < 2 * window:
            basic = np.arange(window)
            inds = np.random.choice(window + 1, cropped_length - window, replace=False)
            offset = np.zeros(window + 1, dtype=np.int64)
            offset[inds] = 1
            offset = np.cumsum(offset)
            inds = basic + offset[:-1]
        else:
            bids = np.array([i * cropped_length // window for i in range(window + 1)])
            bsize = np.diff(bids)
            bst = bids[:window]
            offset = np.random.randint(bsize)
            inds = bst + offset

        inds = inds + bias
        inds = np.clip(inds, 0, T - 1)
        data = data_numpy[:, inds, :, :]
        if data.shape[1] == 0:
            print(cropped_length, bias, valid_size)

    # resize
    data = torch.tensor(data, dtype=torch.float)
    index_t = torch.tensor(inds, dtype=torch.float)
    data = data.permute(2, 3, 0, 1).contiguous().view(V * M, C, len(inds))  # V*M, C, crop_t

    if len(inds) != window:
        data = F.interpolate(data, size=window, mode='linear', align_corners=False)  # V*M, C, T
        index_t = F.interpolate(index_t[None, None, :], size=window, mode='linear', align_corners=False).squeeze()

    data = data.contiguous().view(V, M, C, window).permute(2, 3, 0, 1).contiguous().numpy()
    index_t = 2 * index_t / valid_size - 1
    return data, index_t.numpy()


def scale(data_numpy, scale=0.2, p=0.5):
    if random.random() < p:
        scale = 1 + np.random.uniform(-1, 1, size=(3, 1, 1, 1)) * np.array(scale)
        return data_numpy * scale
    else:
        return data_numpy.copy()


''' AimCLR '''
transform_order = {'ntu': [0, 1, 2, 3, 8, 9, 10, 11, 4, 5, 6, 7, 16, 17, 18, 19, 12, 13, 14, 15, 20, 23, 24, 21, 22]}


def subtract(data_numpy, p=0.5):
    joint = random.randint(0, 24)

    C, T, V, M = data_numpy.shape
    if random.random() < p:
        data_numpy_new = np.zeros((C, T, V, M))
        for i in range(V):
            data_numpy_new[:, :, i, :] = data_numpy[:, :, i, :] - data_numpy[:, :, joint, :]
        return data_numpy_new
    else:
        return data_numpy.copy()


def temporal_flip(data_numpy, index_t, p=0.5):
    C, T, V, M = data_numpy.shape
    if random.random() < p:
        time_range_order = [i for i in range(T)]
        time_range_reverse = list(reversed(time_range_order))
        return data_numpy[:, time_range_reverse, :, :], -index_t
    else:
        return data_numpy.copy(), index_t.copy()


def spatial_flip(data_numpy, p=0.5):
    if random.random() < p:
        index = transform_order['ntu']
        return data_numpy[:, :, index, :]
    else:
        return data_numpy.copy()


def rotate(data_numpy, axis=None, angle=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if angle != None:
        angle_next = random.uniform(-angle, angle)
    else:
        angle_next = random.uniform(-30, 30)

    if random.random() < p:
        temp = data_numpy.copy()
        angle = math.radians(angle_next)
        # x
        if axis_next == 0:
            R = np.array([[1, 0, 0],
                          [0, math.cos(angle), math.sin(angle)],
                          [0, -math.sin(angle), math.cos(angle)]])
        # y
        if axis_next == 1:
            R = np.array([[math.cos(angle), 0, -math.sin(angle)],
                          [0, 1, 0],
                          [math.sin(angle), 0, math.cos(angle)]])
        # z
        if axis_next == 2:
            R = np.array([[math.cos(angle), math.sin(angle), 0],
                          [-math.sin(angle), math.cos(angle), 0],
                          [0, 0, 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def shear(data_numpy, s1=None, s2=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        if s1 != None:
            s1_list = s1
        else:
            s1_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]
        if s2 != None:
            s2_list = s2
        else:
            s2_list = [random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5), random.uniform(-0.5, 0.5)]

        R = np.array([[1, s1_list[0], s2_list[0]],
                      [s1_list[1], 1, s2_list[1]],
                      [s1_list[2], s2_list[2], 1]])
        R = R.transpose()
        temp = np.dot(temp.transpose([1, 2, 3, 0]), R)
        temp = temp.transpose(3, 0, 1, 2)
        return temp
    else:
        return data_numpy.copy()


def drop_axis(data_numpy, axis=None, p=0.5):
    if axis != None:
        axis_next = axis
    else:
        axis_next = random.randint(0, 2)

    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        x_new = np.zeros((T, V, M))
        temp[axis_next] = x_new
        return temp
    else:
        return data_numpy.copy()


def drop_joint(data_numpy, joint_list=None, time_range=None, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape

        if joint_list != None:
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, joint_list)
            joint_list_ = sorted(joint_list_)
        else:
            random_int = random.randint(5, 15)
            all_joints = [i for i in range(V)]
            joint_list_ = random.sample(all_joints, random_int)
            joint_list_ = sorted(joint_list_)

        if time_range != None:
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, time_range)
            time_range_ = sorted(time_range_)
        else:
            random_int = random.randint(16, 32)
            all_frames = [i for i in range(T)]
            time_range_ = random.sample(all_frames, random_int)
            time_range_ = sorted(time_range_)

        x_new = np.zeros((C, len(time_range_), len(joint_list_), M))
        temp2 = temp[:, time_range_, :, :].copy()
        temp2[:, :, joint_list_, :] = x_new
        temp[:, time_range_, :, :] = temp2
        return temp
    else:
        return data_numpy.copy()


def gaussian_noise(data_numpy, mean=0, std=0.05, p=0.5):
    if random.random() < p:
        temp = data_numpy.copy()
        C, T, V, M = data_numpy.shape
        noise = np.random.normal(mean, std, size=(C, T, V, M))
        return temp + noise
    else:
        return data_numpy.copy()


class GaussianBlurConv(nn.Module):
    def __init__(self, channels=3, kernel=15, sigma=[0.1, 2], p=0.5):
        super(GaussianBlurConv, self).__init__()
        self.channels = channels
        self.kernel = kernel
        self.min_max_sigma = sigma
        radius = int(kernel / 2)
        self.kernel_index = np.arange(-radius, radius + 1)
        self.p = p

    def __call__(self, x):
        sigma = random.uniform(self.min_max_sigma[0], self.min_max_sigma[1])
        blur_flter = np.exp(-np.power(self.kernel_index, 2.0) / (2.0 * np.power(sigma, 2.0)))
        kernel = torch.from_numpy(blur_flter).unsqueeze(0).unsqueeze(0)
        # kernel =  kernel.float()
        kernel = kernel.double()
        kernel = kernel.repeat(self.channels, 1, 1, 1)
        self.weight = nn.Parameter(data=kernel, requires_grad=False)

        prob = np.random.random_sample()
        x = torch.from_numpy(x).double()
        if prob < self.p:
            x = x.permute(3, 0, 2, 1)
            x = F.conv2d(x, self.weight, padding=(0, int((self.kernel - 1) / 2)), groups=self.channels)
            x = x.permute(1, -1, -2, 0)

        return x.numpy()


def gaussian_filter(data_numpy, kernel=15, sig_list=[0.1, 2], p=0.5):
    g = GaussianBlurConv(3, kernel, sig_list, p)
    return g(data_numpy)


''' Skeleton AdaIN '''
def skeleton_adain_bone_length(input, ref): # C T V M
    eps = 1e-5
    center = 1
    ref_c = ref[:, :, center, :]

    # joint to bone (joint2bone)
    j2b = joint2bone()
    bone_i = j2b(input) # C T V M
    bone_r = j2b(ref)

    bone_length_i = np.linalg.norm(bone_i, axis=0) # T V M
    bone_length_r = np.linalg.norm(bone_r, axis=0)

    bone_length_scale = (bone_length_r + eps) / (bone_length_i + eps) # T V M
    bone_length_scale = np.expand_dims(bone_length_scale, axis=0) # 1 T V M

    bone_i = bone_i * bone_length_scale

    # bone to joint (bone2joint)
    b2j = bone2joint()
    joint = b2j(bone_i, ref_c)
    return joint


class joint2bone(nn.Module):
    def __init__(self):
        super(joint2bone, self).__init__()
        self.pairs = [(0, 1), (1, 1), (2, 20), (3, 2), (4, 20), (5, 4), (6, 5), (7, 6), (8, 20), (9, 8),
                      (10, 9), (11, 10), (12, 0), (13, 12), (14, 13), (15, 14), (16, 0), (17, 16), (18, 17),
                      (19, 18), (20, 1), (21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, joint):
        bone = np.zeros_like(joint)
        for v1, v2 in self.pairs:
            bone[:, :, v1, :] = joint[:, :, v1, :] - joint[:, :, v2, :]
        return bone


class bone2joint(nn.Module):
    def __init__(self):
        super(bone2joint, self).__init__()
        self.center = 1
        self.pairs_1 = [(0, 1), (20, 1)]
        self.pairs_2 = [(2, 20), (4, 20), (8, 20), (12, 0), (16, 0)]
        self.pairs_3 = [(3, 2), (5, 4), (9, 8), (13, 12), (17, 16)]
        self.pairs_4 = [(6, 5), (10, 9), (14, 13), (18, 17)]
        self.pairs_5 = [(7, 6), (11, 10), (15, 14), (19, 18)]
        self.pairs_6 = [(21, 7), (22, 7), (23, 11), (24, 11)]

    def __call__(self, bone, center):
        joint = np.zeros_like(bone)
        joint[:, :, self.center, :] = center
        for v1, v2 in self.pairs_1:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_2:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_3:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_4:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_5:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        for v1, v2 in self.pairs_6:
            joint[:, :, v1, :] = bone[:, :, v1, :] + joint[:, :, v2, :]
        return joint

def to_motion(input): # C T V M
    C, T, V, M = input.shape
    motion = np.zeros_like(input)
    motion[:, :T - 1] = np.diff(input, axis=1)
    return motion

# 3. Đăng ký module 'tools' vào sys.modules và globals để cả gọi trực tiếp lẫn qua tools.<func> đều hoạt động 100%
tools = types.ModuleType("tools")
for _k, _v in list(globals().items()):
    if not _k.startswith("_"):
        setattr(tools, _k, _v)
sys.modules["tools"] = tools
sys.modules["feeders.tools"] = tools

feeders_mod = types.ModuleType("feeders")
setattr(feeders_mod, "tools", tools)
sys.modules["feeders"] = feeders_mod
print("Data augmentation tools ready and feeders/tools.py written to disk!")



Data augmentation tools ready and feeders/tools.py written to disk!


In [4]:
# NTU RGB+D Feeder
import numpy as np
import random
import csv

from torch.utils.data import Dataset
import sys, types
if '.' not in sys.path:
    sys.path.insert(0, '.')
try:
    from feeders import tools
except Exception:
    if 'tools' not in globals() or tools is None:
        tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))



class Feeder(Dataset):
    def __init__(self, data_path, label_path=None, p_interval=1, split='train', data_type='j',
                 aug_method='z', intra_p=0.5, inter_p=0.0, window_size=-1,
                 debug=False, thres=64, uniform=False, partition=False,
                 concepts_csv=None, remove_null_concept=True):

        self.debug = debug
        self.data_path = data_path
        self.label_path = label_path
        self.split = split
        self.data_type = data_type
        self.aug_method = aug_method
        self.intra_p = intra_p
        self.inter_p = inter_p
        self.window_size = window_size
        self.p_interval = p_interval
        self.thres = thres
        self.uniform = uniform
        self.partition = partition
        self.concepts_csv = concepts_csv
        self.remove_null_concept = remove_null_concept
        self.load_data()
        self.concept_matrix = None
        self.concept_names = None
        if self.concepts_csv:
            self.load_concepts()
        if partition:
            self.right_arm = np.array([7, 8, 22, 23]) - 1
            self.left_arm = np.array([11, 12, 24, 25]) - 1
            self.right_leg = np.array([13, 14, 15, 16]) - 1
            self.left_leg = np.array([17, 18, 19, 20]) - 1
            self.h_torso = np.array([5, 9, 6, 10]) - 1
            self.w_torso = np.array([2, 3, 1, 4]) - 1
            self.new_idx = np.concatenate((self.right_arm, self.left_arm, self.right_leg, self.left_leg, self.h_torso, self.w_torso), axis=-1)
            # except for joint no.21

    def load_concepts(self):
        """Load REASON concept targets without changing the legacy feeder API."""
        with open(self.concepts_csv, 'r', newline='', encoding='utf-8-sig') as handle:
            rows = list(csv.DictReader(handle))

        if not rows or 'action_class' not in rows[0]:
            raise ValueError(f'Invalid concepts CSV: {self.concepts_csv}')

        names = [name for name in rows[0].keys() if name != 'action_class']
        # Nếu CSV có 74 concepts (chưa loại null concepts), lọc bỏ 2 null concepts
        # Nếu CSV đã là 72 concepts chuẩn, giữ nguyên 72 concepts
        if self.remove_null_concept and len(names) == 74:
            names = [name for i, name in enumerate(names) if i not in (50, 65)]
        elif self.remove_null_concept and len(names) > 72:
            names = [name for name in names if name not in ('foot_tap', 'temporal_sequence_seq_cascading')]

        values = []
        for row in rows:
            values.append([float(row[name]) for name in names])
        self.concept_names = names
        self.concept_matrix = np.asarray(values, dtype=np.float32)
        if self.concept_matrix.shape[1] != 72:
            raise ValueError(
                f'Expected 72 concepts after filtering, got {self.concept_matrix.shape[1]}'
            )

    def load_data(self):
        # data: N C V T M
        npz_data = np.load(self.data_path)
        if self.split == 'train':
            self.data = npz_data['x_train']
            self.label = np.where(npz_data['y_train'] > 0)[1]
            self.sample_name = ['train_' + str(i) for i in range(len(self.data))]
        elif self.split == 'test':
            self.data = npz_data['x_test']
            self.label = np.where(npz_data['y_test'] > 0)[1]
            self.sample_name = ['test_' + str(i) for i in range(len(self.data))]
        else:
            raise NotImplementedError('data split only supports train/test')
        N, T, _ = self.data.shape
        self.data = self.data.reshape((N, T, 2, 25, 3)).transpose(0, 4, 1, 3, 2)

    def __len__(self):
        return len(self.label)

    def __iter__(self):
        return self

    def __getitem__(self, index):
        global tools
        if 'tools' not in globals() or tools is None:
            import sys
            try:
                from feeders import tools
            except Exception:
                tools = sys.modules.get('tools', sys.modules.get('feeders.tools', None))

        data_numpy = self.data[index]
        label = self.label[index]
        data_numpy = np.array(data_numpy)
        valid_frame_num = np.sum(data_numpy.sum(0).sum(-1).sum(-1) != 0)
        num_people = np.sum(data_numpy.sum(0).sum(0).sum(0) != 0)

        if self.uniform:
            data_numpy, index_t = tools.valid_crop_uniform(data_numpy, valid_frame_num, self.p_interval,
                                                           self.window_size, self.thres)
        else:
            data_numpy, index_t = tools.valid_crop_resize(data_numpy, valid_frame_num, self.p_interval,
                                                          self.window_size, self.thres)

        if self.split == 'train':
            # intra-instance augmentation
            p = np.random.rand(1)
            if p < self.intra_p:

                if 'a' in self.aug_method:
                    if np.random.rand(1) < 0.5:
                        data_numpy = data_numpy[:, :, :, np.array([1, 0])]
                if 'b' in self.aug_method:
                    if num_people == 2:
                        if np.random.rand(1) < 0.5:
                            axis_next = np.random.randint(0, 1)
                            temp = data_numpy.copy()
                            C, T, V, M = data_numpy.shape
                            x_new = np.zeros((C, T, V))
                            temp[:, :, :, axis_next] = x_new
                            data_numpy = temp

                if '1' in self.aug_method:
                    data_numpy = tools.shear(data_numpy, p=0.5)
                if '2' in self.aug_method:
                    data_numpy = tools.rotate(data_numpy, p=0.5)
                if '3' in self.aug_method:
                    data_numpy = tools.scale(data_numpy, p=0.5)
                if '4' in self.aug_method:
                    data_numpy = tools.spatial_flip(data_numpy, p=0.5)
                if '5' in self.aug_method:
                    data_numpy, index_t = tools.temporal_flip(data_numpy, index_t, p=0.5)
                if '6' in self.aug_method:
                    data_numpy = tools.gaussian_noise(data_numpy, p=0.5)
                if '7' in self.aug_method:
                    data_numpy = tools.gaussian_filter(data_numpy, p=0.5)
                if '8' in self.aug_method:
                    data_numpy = tools.drop_axis(data_numpy, p=0.5)
                if '9' in self.aug_method:
                    data_numpy = tools.drop_joint(data_numpy, p=0.5)

            # inter-instance augmentation
            elif (p < (self.intra_p + self.inter_p)) & (p >= self.intra_p):
                adain_idx = random.choice(np.where(self.label == label)[0])
                data_adain = self.data[adain_idx]
                data_adain = np.array(data_adain)
                f_num = np.sum(data_adain.sum(0).sum(-1).sum(-1) != 0)
                t_idx = np.round((index_t + 1) * f_num / 2).astype(np.int64)
                data_adain = data_adain[:, t_idx]
                data_numpy = tools.skeleton_adain_bone_length(data_numpy, data_adain)

            else:
                data_numpy = data_numpy.copy()

        # modality
        if self.data_type == 'b':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
        elif self.data_type == 'jm':
            data_numpy = tools.to_motion(data_numpy)
        elif self.data_type == 'bm':
            j2b = tools.joint2bone()
            data_numpy = j2b(data_numpy)
            data_numpy = tools.to_motion(data_numpy)
        else:
            data_numpy = data_numpy.copy()

        if self.partition:
            data_numpy = data_numpy[:, :, self.new_idx]

        if self.concept_matrix is not None:
            if label >= len(self.concept_matrix):
                raise IndexError(f'Concept CSV has no row for action label {label}')
            return data_numpy, index_t, label, index, self.concept_matrix[label]
        return data_numpy, index_t, label, index

    def top_k(self, score, top_k):
        rank = score.argsort()
        hit_top_k = [l in rank[i, -top_k:] for i, l in enumerate(self.label)]
        return sum(hit_top_k) * 1.0 / len(hit_top_k)


def import_class(name):
    components = name.split('.')
    mod = __import__(components[0])
    for comp in components[1:]:
        mod = getattr(mod, comp)
    return mod




## 🧠 Block 3: SkateFormer Perception Backbone


In [5]:
# SkateFormer Backbone Architecture
from typing import Type, Tuple, Optional, Set, List, Union

import math
import numpy as np
import torch
import torch.nn as nn
from timm.models.layers import drop_path, trunc_normal_, Mlp, DropPath, create_act_layer, get_norm_act_layer, create_conv2d

''' Partition and Reverse '''


def type_1_partition(input, partition_size):  # partition_size = [N, L]
    B, C, T, V = input.shape
    partitions = input.view(B, C, T // partition_size[0], partition_size[0], V // partition_size[1], partition_size[1])
    partitions = partitions.permute(0, 2, 4, 3, 5, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_1_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 1, 3, 2, 4).contiguous().view(B, -1, T, V)
    return output


def type_2_partition(input, partition_size):  # partition_size = [N, K]
    B, C, T, V = input.shape
    partitions = input.view(B, C, T // partition_size[0], partition_size[0], partition_size[1], V // partition_size[1])
    partitions = partitions.permute(0, 2, 5, 3, 4, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_2_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 1, 3, 4, 2).contiguous().view(B, -1, T, V)
    return output


def type_3_partition(input, partition_size):  # partition_size = [M, L]
    B, C, T, V = input.shape
    partitions = input.view(B, C, partition_size[0], T // partition_size[0], V // partition_size[1], partition_size[1])
    partitions = partitions.permute(0, 3, 4, 2, 5, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_3_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 3, 1, 2, 4).contiguous().view(B, -1, T, V)
    return output


def type_4_partition(input, partition_size):  # partition_size = [M, K]
    B, C, T, V = input.shape
    partitions = input.view(B, C, partition_size[0], T // partition_size[0], partition_size[1], V // partition_size[1])
    partitions = partitions.permute(0, 3, 5, 2, 4, 1).contiguous().view(-1, partition_size[0], partition_size[1], C)
    return partitions


def type_4_reverse(partitions, original_size, partition_size):  # original_size = [T, V]
    T, V = original_size
    B = int(partitions.shape[0] / (T * V / partition_size[0] / partition_size[1]))
    output = partitions.view(B, T // partition_size[0], V // partition_size[1], partition_size[0], partition_size[1], -1)
    output = output.permute(0, 5, 3, 1, 4, 2).contiguous().view(B, -1, T, V)
    return output


''' 1D relative positional bias: B_{h}^{t} '''


def get_relative_position_index_1d(T):
    coords = torch.stack(torch.meshgrid([torch.arange(T)]))
    coords_flatten = torch.flatten(coords, 1)
    relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
    relative_coords = relative_coords.permute(1, 2, 0).contiguous()
    relative_coords[:, :, 0] += T - 1
    return relative_coords.sum(-1)


''' MSA '''


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, in_channels, rel_type, num_heads=32, partition_size=(1, 1), attn_drop=0., rel=True):
        super(MultiHeadSelfAttention, self).__init__()
        self.in_channels = in_channels
        self.rel_type = rel_type
        self.num_heads = num_heads
        self.partition_size = partition_size
        self.scale = num_heads ** -0.5
        self.attn_area = partition_size[0] * partition_size[1]
        self.attn_drop = nn.Dropout(p=attn_drop)
        self.softmax = nn.Softmax(dim=-1)
        self.rel = rel

        if self.rel:
            if self.rel_type == 'type_1' or self.rel_type == 'type_3':
                self.relative_position_bias_table = nn.Parameter(torch.zeros((2 * partition_size[0] - 1), num_heads))
                self.register_buffer("relative_position_index", get_relative_position_index_1d(partition_size[0]))
                trunc_normal_(self.relative_position_bias_table, std=.02)
                self.ones = torch.ones(partition_size[1], partition_size[1], num_heads)
            elif self.rel_type == 'type_2' or self.rel_type == 'type_4':
                self.relative_position_bias_table = nn.Parameter(
                    torch.zeros((2 * partition_size[0] - 1), partition_size[1], partition_size[1], num_heads))
                self.register_buffer("relative_position_index", get_relative_position_index_1d(partition_size[0]))
                trunc_normal_(self.relative_position_bias_table, std=.02)

    def _get_relative_positional_bias(self):
        if self.rel_type == 'type_1' or self.rel_type == 'type_3':
            relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
                self.partition_size[0], self.partition_size[0], -1)
            relative_position_bias = relative_position_bias.unsqueeze(1).unsqueeze(3).repeat(1, self.partition_size[1], 1, self.partition_size[1], 1, 1).view(self.attn_area, self.attn_area, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            return relative_position_bias.unsqueeze(0)
        elif self.rel_type == 'type_2' or self.rel_type == 'type_4':
            relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(self.partition_size[0], self.partition_size[0], self.partition_size[1], self.partition_size[1], -1)
            relative_position_bias = relative_position_bias.permute(0, 2, 1, 3, 4).contiguous().view(self.attn_area, self.attn_area, -1)
            relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()
            return relative_position_bias.unsqueeze(0)

    def forward(self, input):
        B_, N, C = input.shape
        qkv = input.reshape(B_, N, 3, self.num_heads, -1).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        q = q * self.scale
        attn = q @ k.transpose(-2, -1)
        if self.rel:
            attn = attn + self._get_relative_positional_bias()
        attn = self.softmax(attn)
        attn = self.attn_drop(attn)
        output = (attn @ v).transpose(1, 2).reshape(B_, N, -1)
        return output


''' SkateFormer Block '''


class SkateFormerBlock(nn.Module):
    def __init__(self, in_channels, num_points=50, kernel_size=7, num_heads=32,
                 type_1_size=(8, 8), type_2_size=(8, 12), type_3_size=(8, 8), type_4_size=(8, 12),
                 attn_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
                 act_layer=nn.GELU, norm_layer=nn.LayerNorm):
        super(SkateFormerBlock, self).__init__()
        self.type_1_size = type_1_size
        self.type_2_size = type_2_size
        self.type_3_size = type_3_size
        self.type_4_size = type_4_size
        self.partition_function = [type_1_partition, type_2_partition, type_3_partition, type_4_partition]
        self.reverse_function = [type_1_reverse, type_2_reverse, type_3_reverse, type_4_reverse]
        self.partition_size = [type_1_size, type_2_size, type_3_size, type_4_size]
        self.rel_type = ['type_1', 'type_2', 'type_3', 'type_4']

        self.norm_1 = norm_layer(in_channels)
        self.mapping = nn.Linear(in_features=in_channels, out_features=2 * in_channels, bias=True)
        self.gconv = nn.Parameter(torch.zeros(num_heads // (2 * 2), num_points, num_points))
        trunc_normal_(self.gconv, std=.02)
        self.tconv = nn.Conv2d(in_channels // (2 * 2), in_channels // (2 * 2), kernel_size=(kernel_size, 1),
                               padding=((kernel_size - 1) // 2, 0), groups=num_heads // (2 * 2))

        # Attention layers
        attention = []
        for i in range(len(self.partition_function)):
            attention.append(
                MultiHeadSelfAttention(in_channels=in_channels // (len(self.partition_function) * 2),
                                       rel_type=self.rel_type[i],
                                       num_heads=num_heads // (len(self.partition_function) * 2),
                                       partition_size=self.partition_size[i], attn_drop=attn_drop, rel=rel))
        self.attention = nn.ModuleList(attention)
        self.proj = nn.Linear(in_features=in_channels, out_features=in_channels, bias=True)
        self.proj_drop = nn.Dropout(p=drop)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm_2 = norm_layer(in_channels)
        self.mlp = Mlp(in_features=in_channels, hidden_features=int(mlp_ratio * in_channels),
                       act_layer=act_layer, drop=drop)

    def forward(self, input):
        B, C, T, V = input.shape

        # Partition
        input = input.permute(0, 2, 3, 1).contiguous()
        skip = input

        f = self.mapping(self.norm_1(input)).permute(0, 3, 1, 2).contiguous()

        f_conv, f_attn = torch.split(f, [C // 2, 3 * C // 2], dim=1)
        y = []

        # G-Conv
        split_f_conv = torch.chunk(f_conv, 2, dim=1)
        y_gconv = []
        split_f_gconv = torch.chunk(split_f_conv[0], self.gconv.shape[0], dim=1)
        for i in range(self.gconv.shape[0]):
            z = torch.einsum('n c t u, v u -> n c t v', split_f_gconv[i], self.gconv[i])
            y_gconv.append(z)
        y.append(torch.cat(y_gconv, dim=1))  # N C T V

        # T-Conv
        y.append(self.tconv(split_f_conv[1]))

        # Skate-MSA
        split_f_attn = torch.chunk(f_attn, len(self.partition_function), dim=1)

        for i in range(len(self.partition_function)):
            C = split_f_attn[i].shape[1]
            input_partitioned = self.partition_function[i](split_f_attn[i], self.partition_size[i])
            input_partitioned = input_partitioned.view(-1, self.partition_size[i][0] * self.partition_size[i][1], C)
            y.append(self.reverse_function[i](self.attention[i](input_partitioned), (T, V), self.partition_size[i]))

        output = self.proj(torch.cat(y, dim=1).permute(0, 2, 3, 1).contiguous())
        output = self.proj_drop(output)
        output = skip + self.drop_path(output)

        # Feed Forward
        output = output + self.drop_path(self.mlp(self.norm_2(output)))
        output = output.permute(0, 3, 1, 2).contiguous()
        return output


''' Downsampling '''


class PatchMergingTconv(nn.Module):
    def __init__(self, dim_in, dim_out, kernel_size=7, stride=2, dilation=1):
        super().__init__()
        self.dim_in = dim_in
        self.dim_out = dim_out
        pad = (kernel_size + (kernel_size - 1) * (dilation - 1) - 1) // 2
        self.reduction = nn.Conv2d(dim_in, dim_out, kernel_size=(kernel_size, 1), padding=(pad, 0), stride=(stride, 1),
                                   dilation=(dilation, 1))
        self.bn = nn.BatchNorm2d(dim_out)

    def forward(self, x):
        x = self.bn(self.reduction(x))
        return x


''' SkateFormer Block with Downsampling '''


class SkateFormerBlockDS(nn.Module):
    def __init__(
            self, in_channels, out_channels, num_points=50, kernel_size=7, downscale=False, num_heads=32,
            type_1_size=(8, 8), type_2_size=(8, 12), type_3_size=(8, 8), type_4_size=(8, 12),
            attn_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
            act_layer=nn.GELU, norm_layer_transformer=nn.LayerNorm):
        super(SkateFormerBlockDS, self).__init__()

        if downscale:
            self.downsample = PatchMergingTconv(in_channels, out_channels, kernel_size=kernel_size)
        else:
            self.downsample = None

        self.transformer = SkateFormerBlock(
            in_channels=out_channels,
            num_points=num_points,
            kernel_size=kernel_size,
            num_heads=num_heads,
            type_1_size=type_1_size,
            type_2_size=type_2_size,
            type_3_size=type_3_size,
            type_4_size=type_4_size,
            attn_drop=attn_drop,
            drop=drop,
            rel=rel,
            drop_path=drop_path,
            mlp_ratio=mlp_ratio,
            act_layer=act_layer,
            norm_layer=norm_layer_transformer,
        )

    def forward(self, input):
        if self.downsample is not None:
            output = self.transformer(self.downsample(input))
        else:
            output = self.transformer(input)
        return output


''' SkateFormer Stage '''


class SkateFormerStage(nn.Module):
    def __init__(
            self, depth, in_channels, out_channels, first_depth=False,
            num_points=50, kernel_size=7, num_heads=32,
            type_1_size=(8, 8), type_2_size=(8, 12), type_3_size=(8, 8), type_4_size=(8, 12),
            attn_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
            act_layer=nn.GELU, norm_layer_transformer=nn.LayerNorm):
        super(SkateFormerStage, self).__init__()
        blocks = []
        for index in range(depth):
            blocks.append(
                SkateFormerBlockDS(
                    in_channels=in_channels if index == 0 else out_channels,
                    out_channels=out_channels,
                    num_points=num_points,
                    kernel_size=kernel_size,
                    downscale=((index == 0) and not first_depth),
                    num_heads=num_heads,
                    type_1_size=type_1_size,
                    type_2_size=type_2_size,
                    type_3_size=type_3_size,
                    type_4_size=type_4_size,
                    attn_drop=attn_drop,
                    drop=drop,
                    rel=rel,
                    drop_path=drop_path if isinstance(drop_path, float) else drop_path[index],
                    mlp_ratio=mlp_ratio,
                    act_layer=act_layer,
                    norm_layer_transformer=norm_layer_transformer))
        self.blocks = nn.ModuleList(blocks)

    def forward(self, input):
        output = input
        for block in self.blocks:
            output = block(output)
        return output


''' SkateFormer '''


class SkateFormer(nn.Module):
    def __init__(self, in_channels=3, depths=(2, 2, 2, 2), channels=(96, 192, 192, 192), num_classes=60,
                 embed_dim=64, num_people=2, num_frames=64, num_points=50, kernel_size=7, num_heads=32,
                 type_1_size=(8, 8), type_2_size=(8, 12), type_3_size=(8, 8), type_4_size=(8, 12),
                 attn_drop=0., head_drop=0., drop=0., rel=True, drop_path=0., mlp_ratio=4.,
                 act_layer=nn.GELU, norm_layer_transformer=nn.LayerNorm, index_t=True, global_pool='avg'):

        super(SkateFormer, self).__init__()

        assert len(depths) == len(channels), "For each stage a channel dimension must be given."
        assert global_pool in ["avg", "max"], f"Only avg and max is supported but {global_pool} is given"
        self.num_classes: int = num_classes
        self.head_drop = head_drop
        self.index_t = index_t
        self.embed_dim = embed_dim

        if self.head_drop != 0:
            self.dropout = nn.Dropout(p=self.head_drop)
        else:
            self.dropout = None

        stem = []
        stem.append(nn.Conv2d(in_channels=in_channels, out_channels=2 * in_channels, kernel_size=(1, 1), stride=(1, 1),
                              padding=(0, 0)))
        stem.append(act_layer())
        stem.append(
            nn.Conv2d(in_channels=2 * in_channels, out_channels=3 * in_channels, kernel_size=(1, 1), stride=(1, 1),
                      padding=(0, 0)))
        stem.append(act_layer())
        stem.append(nn.Conv2d(in_channels=3 * in_channels, out_channels=embed_dim, kernel_size=(1, 1), stride=(1, 1),
                              padding=(0, 0)))
        self.stem = nn.ModuleList(stem)

        if self.index_t:
            self.joint_person_embedding = nn.Parameter(torch.zeros(embed_dim, num_points * num_people))
            trunc_normal_(self.joint_person_embedding, std=.02)
        else:
            self.joint_person_temporal_embedding = nn.Parameter(
                torch.zeros(1, embed_dim, num_frames, num_points * num_people))
            trunc_normal_(self.joint_person_temporal_embedding, std=.02)

        # Init blocks
        drop_path = torch.linspace(0.0, drop_path, sum(depths)).tolist()
        stages = []
        for index, (depth, channel) in enumerate(zip(depths, channels)):
            stages.append(
                SkateFormerStage(
                    depth=depth,
                    in_channels=embed_dim if index == 0 else channels[index - 1],
                    out_channels=channel,
                    first_depth=index == 0,
                    num_points=num_points * num_people,
                    kernel_size=kernel_size,
                    num_heads=num_heads,
                    type_1_size=type_1_size,
                    type_2_size=type_2_size,
                    type_3_size=type_3_size,
                    type_4_size=type_4_size,
                    attn_drop=attn_drop,
                    drop=drop,
                    rel=rel,
                    drop_path=drop_path[sum(depths[:index]):sum(depths[:index + 1])],
                    mlp_ratio=mlp_ratio,
                    act_layer=act_layer,
                    norm_layer_transformer=norm_layer_transformer
                )
            )
        self.stages = nn.ModuleList(stages)
        self.global_pool: str = global_pool
        self.head = nn.Linear(channels[-1], num_classes)

    @torch.jit.ignore
    def no_weight_decay(self):
        nwd = set()
        for n, _ in self.named_parameters():
            if "relative_position_bias_table" in n:
                nwd.add(n)
        return nwd

    def reset_classifier(self, num_classes, global_pool=None):
        self.num_classes: int = num_classes
        if global_pool is not None:
            self.global_pool = global_pool
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()

    def forward_features(self, input):
        output = input
        for stage in self.stages:
            output = stage(output)
        return output

    def forward_head(self, input, pre_logits=False):
        if self.global_pool == "avg":
            input = input.mean(dim=(2, 3))
        elif self.global_pool == "max":
            input = torch.amax(input, dim=(2, 3))
        if self.dropout is not None:
            input = self.dropout(input)
        return input if pre_logits else self.head(input)

    def forward_features_from_input(self, input, index_t=None):
        """Return the final feature map before global pooling/classification.

        This is shared by the original classifier and REASON-SkateFormer so
        the pretrained SkateFormer stem and temporal embedding are identical
        in both paths.
        """
        B, C, T, V, M = input.shape

        output = input.permute(0, 1, 2, 4, 3).contiguous().view(B, C, T, -1)  # [B, C, T, M * V]
        for layer in self.stem:
            output = layer(output)
        if self.index_t:
            if index_t is None:
                index_t = torch.linspace(-1., 1., T, device=output.device, dtype=output.dtype)
                index_t = index_t.unsqueeze(0).expand(B, T)
            te = torch.zeros(B, T, self.embed_dim).to(output.device)  # B, T, C
            div_term = torch.exp(
                (torch.arange(0, self.embed_dim, 2, dtype=torch.float) * -(math.log(10000.0) / self.embed_dim))).to(
                output.device)
            te[:, :, 0::2] = torch.sin(index_t.unsqueeze(-1).float() * div_term)
            te[:, :, 1::2] = torch.cos(index_t.unsqueeze(-1).float() * div_term)
            output = output + torch.einsum('b t c, c v -> b c t v', te, self.joint_person_embedding)
        else:
            output = output + self.joint_person_temporal_embedding
        return self.forward_features(output)

    def forward(self, input, index_t=None):
        output = self.forward_features_from_input(input, index_t=index_t)
        output = self.forward_head(output)
        return output


def SkateFormer_(
    type_1_size=(8, 8),
    type_2_size=(8, 12),
    type_3_size=(8, 8),
    type_4_size=(8, 12),
    index_t=True,
    **kwargs
):
    return SkateFormer(
        depths=(2, 2, 2, 2),
        channels=(96, 192, 192, 192),
        embed_dim=96,
        type_1_size=type_1_size,
        type_2_size=type_2_size,
        type_3_size=type_3_size,
        type_4_size=type_4_size,
        index_t=index_t,
        **kwargs
    )




ModuleNotFoundError: No module named 'torchvision'

## ⚡ Block 4: Tầng Logic CRL


In [ ]:
# 1. LiteralLayer & BinarizeSTE
"""LiteralLayer: Chuyển đổi concept logits thành literals nhị phân.

Module này thực hiện bước đầu tiên trong luận chứng logic (CRL pipeline):
  z^t ∈ R^(B × 72)  →  l^t ∈ R^(B × 144)

Hai chế độ:
  - 'ste':  Dùng Straight-Through Estimator nhị phân (mặc định: Pure STE, temperature annealing 2.0 → 10.0)
  - 'soft': Dùng sigmoid liên tục (hỗ trợ warm-up / phân tích liên tục nếu cần)

Nguyên tắc bất biến:
  z^t (concept logits) LUÔN là số thực liên tục.
  Binarization CHỈ xảy ra khi tạo l^t để truyền vào CRL.
"""

import torch
import torch.nn as nn


class BinarizeSTE(torch.autograd.Function):
    """Straight-Through Estimator (STE) cho binarization.

    Forward:  hard = (z > 0).float()  [nhị phân {0, 1}]
    Backward: gradient đi thẳng qua sigmoid liên tục (không bị chặn bởi hard step)
    """

    @staticmethod
    def forward(ctx, z: torch.Tensor, temperature: float) -> torch.Tensor:
        # Lưu sigmoid liên tục để dùng trong backward
        soft = torch.sigmoid(temperature * z)
        ctx.save_for_backward(soft)
        ctx.temperature = temperature
        # Output nhị phân
        hard = (z > 0.0).float()
        # STE trick: hard + soft - soft.detach() → forward = hard, backward ≈ d(soft)/dz
        return hard + soft - soft.detach()

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        (soft,) = ctx.saved_tensors
        # Gradient của sigmoid: soft * (1 - soft) * temperature
        grad_z = grad_output * soft * (1.0 - soft) * ctx.temperature
        return grad_z, None  # None cho temperature (không học được)


def binarize_ste(z: torch.Tensor, temperature: float = 2.0) -> torch.Tensor:
    """Hàm tiện ích gọi BinarizeSTE."""
    return BinarizeSTE.apply(z, temperature)


class RMSNorm(nn.Module):
    """Root Mean Square Layer Normalization (tương tự REASON / PyTorch RMSNorm).

    Chuẩn hóa concept logits z^t để kháng vanishing/exploding gradient trước logic layer.
    """

    def __init__(self, normalized_shape: int, eps: float = 1e-8, elementwise_affine: bool = True):
        super().__init__()
        self.normalized_shape = (int(normalized_shape),)
        self.eps = float(eps)
        self.elementwise_affine = bool(elementwise_affine)
        if self.elementwise_affine:
            self.weight = nn.Parameter(torch.ones(self.normalized_shape))
        else:
            self.register_parameter("weight", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dims = tuple(range(-len(self.normalized_shape), 0))
        rms = x.pow(2).mean(dim=dims, keepdim=True).add(self.eps).sqrt()
        y = x / rms
        if self.weight is not None:
            y = y * self.weight
        return y

    def extra_repr(self) -> str:
        return f"normalized_shape={self.normalized_shape}, eps={self.eps}, elementwise_affine={self.elementwise_affine}"


class ScaledSoftsign(nn.Module):
    """Softsign with adjustable bounds - gentlest saturation, best gradient preservation (từ REASON).

    scaled_softsign(x) = scale * x / (beta + |x|), output ∈ (-scale, scale).
    Không bão hòa nhanh như sigmoid/tanh, gradient mượt hơn ở vùng xa 0.
    """

    def __init__(self, scale: float = 3.0, beta: float = 1.0):
        super().__init__()
        self.scale = float(scale)
        self.beta = float(beta)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.scale * x / (self.beta + torch.abs(x))

    def extra_repr(self) -> str:
        return f"scale={self.scale}, beta={self.beta}"


class LiteralLayer(nn.Module):
    """Chuyển đổi concept logits z^t thành literals nhị phân hoặc liên tục.

    Quy trình tiền xử lý theo REASON:
      z^t ∈ R^(B × n_concepts)
        → RMSNorm(n_concepts)
        → ScaledSoftsign() ∈ (-3, 3)
        → STE binarization hoặc Sigmoid
        → Literals l = [p, 1 - p] ∈ R^(B × 2*n_concepts)

    Args:
        n_concepts: Số lượng concepts (mặc định 72).
        mode: 'ste' (mặc định cho Pure STE) hoặc 'soft'.
        temperature: Nhiệt độ cho STE (annealing từ 2.0 → 10.0 xuyên suốt training).
        scale: Scale factor cho ScaledSoftsign (mặc định 3.0).
        beta: Beta parameter cho ScaledSoftsign (mặc định 1.0).
    """

    def __init__(
        self,
        n_concepts: int = 72,
        mode: str = "ste",
        temperature: float = 2.0,
        scale: float = 3.0,
        beta: float = 1.0,
    ):
        super().__init__()
        self.n_concepts = int(n_concepts)
        self._mode = mode
        self.temperature = float(temperature)

        # REASON Mechanism 1: Preprocessing Input
        self.norm = RMSNorm(self.n_concepts)
        self.scaled_softsign = ScaledSoftsign(scale=scale, beta=beta)

    @property
    def mode(self) -> str:
        return self._mode

    @mode.setter
    def mode(self, value: str):
        assert value in ("soft", "ste"), f"mode phải là 'soft' hoặc 'ste', nhận được: {value!r}"
        self._mode = value

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        """Tính literals từ concept logits.

        Args:
            z: Concept logits [B, n_concepts] (LUÔN là số thực liên tục)

        Returns:
            l: Literals [B, 2 * n_concepts] (cả dương lẫn phủ định)
        """
        # Bước 1: RMSNorm + ScaledSoftsign tiền xử lý logits
        z_prep = self.norm(z)
        z_prep = self.scaled_softsign(z_prep)

        # Bước 2: Binarize hoặc Soft Sigmoid
        if self._mode == "soft":
            p = torch.sigmoid(z_prep)                  # [B, n_concepts] ∈ (0, 1)
            l = torch.cat([p, 1.0 - p], dim=-1)       # [B, 2 * n_concepts]
        else:  # 'ste'
            c_ste = binarize_ste(z_prep, self.temperature)   # ≈ {0, 1}, grad qua sigmoid
            l = torch.cat([c_ste, 1.0 - c_ste], dim=-1)      # [B, 2 * n_concepts]
        return l

    def extra_repr(self) -> str:
        return f"n_concepts={self.n_concepts}, mode={self._mode!r}, temperature={self.temperature}"





In [ ]:
# 2. ConjunctionLayer (L1 Part-Grouped & L2a AND2)
"""ConjunctionLayer: Tầng AND rules cho CRL pipeline.

Module này thực hiện hai tầng hội:
  L1: ConjunctionLayer  (AND, part-grouped) — 148 literals → 256 rules
  L2a: ConjunctionLayer2 (AND2, fully-connected) — 256 rules → 128 compound rules

Công thức Log-Space Product (chống vanishing gradient):
  r_j = exp( Σ_i w_ij · log(l_i + ε) )
       = ∏_i (l_i + ε)^w_ij

  Trong đó w_ij = sigmoid(θ_ij) ∈ [0, 1]  (trọng số học được)

Part-Grouped Architecture (L1):
  Mỗi nhóm rule chỉ kết nối với literals của nhóm bộ phận cơ thể tương ứng.
  Group mask là tensor nhị phân cố định (register_buffer).

  Nhóm bộ phận | Concepts (index) | Literals | Rules
  -------------|------------------|----------|------
  Head         |  0 – 5           |  12      |  32
  Hand         |  6 – 16          |  22      |  48
  Arm          | 17 – 27          |  22      |  48
  Hip          | 28 – 35          |  16      |  32
  Leg          | 36 – 44          |  18      |  32
  Foot         | 45 – 51          |  14      |  24
  Temporal     | 53 – 73          |  42      |  40
  Total        |                  | 148      | 256
"""

import torch
import torch.nn as nn

# ─── Hằng số cấu hình Part-Grouped ───────────────────────────────────────────

_DEFAULT_N_CONCEPTS = 72   # 72 concepts sau khi loại bỏ 2 null concepts (52 spatial + 20 temporal)
_DEFAULT_N_LITERALS = 144  # 2 × 72
_N_RULES_L1         = 256  # Σ rules của các nhóm
_N_RULES_L2A        = 128  # AND2 (L2a)

_EPS = 1e-6  # numerical stability cho log


def get_part_groups(n_concepts: int) -> list:
    """Trả về danh sách phân nhóm bộ phận cơ thể phù hợp với số concepts."""
    if n_concepts == 72:
        # Feeder mặc định remove_null_concept=True: 52 spatial + 20 temporal = 72
        return [
            ("head",     0,   6,   32),
            ("hand",     6,  17,   48),
            ("arm",     17,  28,   48),
            ("hip",     28,  36,   32),
            ("leg",     36,  45,   32),
            ("foot",    45,  52,   24),
            ("temporal",52,  72,   40),
        ]
    elif n_concepts == 74:
        # Full ontology chưa loại bỏ null concepts: 53 spatial + 21 temporal = 74
        return [
            ("head",     0,   6,   32),
            ("hand",     6,  17,   48),
            ("arm",     17,  28,   48),
            ("hip",     28,  36,   32),
            ("leg",     36,  45,   32),
            ("foot",    45,  53,   24),
            ("temporal",53,  74,   40),
        ]
    else:
        # Fallback chia đều cho các nhóm
        step = max(1, n_concepts // 7)
        return [
            ("group_0", 0, step, 32),
            ("group_1", step, 2 * step, 48),
            ("group_2", 2 * step, 3 * step, 48),
            ("group_3", 3 * step, 4 * step, 32),
            ("group_4", 4 * step, 5 * step, 32),
            ("group_5", 5 * step, 6 * step, 24),
            ("group_6", 6 * step, n_concepts, 40),
        ]


def _build_part_group_mask(n_literals: int = _DEFAULT_N_LITERALS, n_rules: int = _N_RULES_L1) -> torch.Tensor:
    """Xây dựng group mask nhị phân [n_literals, n_rules].

    Mask[i, j] = 1.0 nếu literal i thuộc nhóm bộ phận của rule j, 0.0 nếu không.
    Literals của concept c: index c (dương) và c + n_concepts (phủ định).
    """
    mask = torch.zeros(n_literals, n_rules)
    rule_start = 0
    n_concepts = n_literals // 2
    groups = get_part_groups(n_concepts)

    for (name, c_start, c_end, n_group_rules) in groups:
        rule_end = rule_start + n_group_rules
        for c in range(c_start, min(c_end, n_concepts)):
            # Literal dương: index c
            mask[c, rule_start:rule_end] = 1.0
            # Literal phủ định: index c + n_concepts
            mask[c + n_concepts, rule_start:rule_end] = 1.0
        rule_start = rule_end

    return mask


class Product(torch.autograd.Function):
    """Tensor product function with Łukasiewicz T-norm approximation (từ REASON).

    y = -1.0 / (-1.0 + torch.sum(torch.log(X), dim=1))
    Kháng underflow hoàn toàn vì mẫu số không thể triệt tiêu thành 0 như exp(-inf).
    """

    @staticmethod
    def forward(ctx, X: torch.Tensor) -> torch.Tensor:
        # X: [B, input_dim, output_dim]
        X_clamped = X.clamp(min=1e-6)
        log_X = torch.log(X_clamped)
        sum_log_X = torch.sum(log_X, dim=1)  # [B, output_dim]
        y = -1.0 / (-1.0 + sum_log_X)
        ctx.save_for_backward(X_clamped, y)
        return y

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor) -> torch.Tensor:
        X_clamped, y = ctx.saved_tensors
        # grad_output: [B, output_dim]
        # grad_input: [B, input_dim, output_dim]
        grad_input = grad_output.unsqueeze(1) * (y.unsqueeze(1) ** 2 / (X_clamped + 1e-10))
        return grad_input


class GradGraft(torch.autograd.Function):
    """Gradient Grafting (STE cho Logic Layers từ REASON).

    Forward:  Dùng binary output (tường minh, phục vụ suy luận logic tượng trưng).
    Backward: Chuyển gradient qua nhánh liên tục để tối ưu hóa trọng số.
    """

    @staticmethod
    def forward(ctx, X: torch.Tensor, Y: torch.Tensor) -> torch.Tensor:
        return X

    @staticmethod
    def backward(ctx, grad_output: torch.Tensor):
        return None, grad_output.clone()


class ConjunctionLayer(nn.Module):
    """L1 AND layer (Primitive Full-Body Pose Rules).

    Cải tiến theo 6 cơ chế REASON:
      1. W ∈ [0, 1] trực tiếp (khởi tạo 0.5 * rand, clip sau mỗi optimizer step)
      2. Łukasiewicz Product T-norm: Product.apply(1 - (1 - l) * W)
      3. Full-body connections (mặc định part_grouped=False)
      4. GradGraft: forward binary, backward continuous
      5. L1 Sparsity Penalty: l1_norm() và sparsity_loss()

    Input:  l ∈ R^(B × n_literals)     — literals từ LiteralLayer
    Output: r_and1 ∈ R^(B × n_rules)   — conjunction rule activations
    """

    def __init__(
        self,
        n_literals: int = _DEFAULT_N_LITERALS,
        n_rules: int = _N_RULES_L1,
        part_grouped: bool = False,
        init_range: float = 0.5,
        grad_graft: bool = False,
        eps: float = _EPS,
    ):
        super().__init__()
        self.n_literals = int(n_literals)
        self.n_rules = int(n_rules)
        self.part_grouped = bool(part_grouped)
        self.grad_graft = bool(grad_graft)
        self.eps = float(eps)

        # Trọng số W ∈ [0, 1] trực tiếp (REASON Mechanism 3)
        self.W = nn.Parameter(float(init_range) * torch.rand(self.n_literals, self.n_rules))

        if self.part_grouped:
            mask = _build_part_group_mask(self.n_literals, self.n_rules)
            self.register_buffer("group_mask", mask)
        else:
            self.group_mask = None

    @property
    def weight_logit(self) -> torch.Tensor:
        """Hỗ trợ tương thích ngược cho code cũ truy cập weight_logit."""
        return torch.logit(self.W.data.clamp(1e-4, 1.0 - 1e-4))

    def _effective_weights(self) -> torch.Tensor:
        """Trọng số thực tế W ∈ [0, 1]."""
        w = self.W.clamp(0.0, 1.0)
        if self.group_mask is not None:
            w = w * self.group_mask
        return w

    def clip(self):
        """Clip trọng số vào đoạn [0, 1] sau optimizer step (REASON Mechanism 3)."""
        self.W.data.clamp_(0.0, 1.0)

    def l1_norm(self) -> torch.Tensor:
        """Tổng L1 norm cho sparsity penalty (REASON Mechanism 5)."""
        return torch.sum(self._effective_weights())

    def sparsity_loss(self) -> torch.Tensor:
        """Mean sparsity loss."""
        return self._effective_weights().mean()

    def continuous_forward(self, l: torch.Tensor) -> torch.Tensor:
        """Product T-norm liên tục (Łukasiewicz approximation)."""
        w = self._effective_weights()                       # [n_literals, n_rules]
        X = 1.0 - (1.0 - l).unsqueeze(-1) * w               # [B, n_literals, n_rules]
        return Product.apply(X)                             # [B, n_rules]

    @torch.no_grad()
    def binarized_forward(self, l: torch.Tensor) -> torch.Tensor:
        """Binarized rule execution với W > 0.5."""
        w = self._effective_weights()
        Wb = (w > 0.5).float()
        return torch.prod(1.0 - (1.0 - l).unsqueeze(-1) * Wb, dim=1)

    def forward(self, l: torch.Tensor, continuous: bool = None) -> torch.Tensor:
        """Forward pass.

        Args:
            l: Literals [B, n_literals]
            continuous: Nếu True, dùng continuous forward. Nếu None, dùng not self.grad_graft.
        """
        res_tilde = self.continuous_forward(l)
        use_cont = continuous if continuous is not None else (not self.grad_graft)
        if use_cont:
            return res_tilde
        res_bar = self.binarized_forward(l)
        return GradGraft.apply(res_bar, res_tilde)

    def extra_repr(self) -> str:
        return (
            f"n_literals={self.n_literals}, n_rules={self.n_rules}, "
            f"part_grouped={self.part_grouped}, grad_graft={self.grad_graft}"
        )


class ConjunctionLayer2(nn.Module):
    """L2a AND2 layer: Hội cấp 2 của các AND rules (L1 -> L2a).

    Mô tả các hành động đòi hỏi nhiều bộ phận phối hợp đồng thời.
    Sử dụng Łukasiewicz Product T-norm và W ∈ [0, 1] trực tiếp.

    Input:  r_and ∈ R^(B × n_rules_l1)   — L1 conjunction rule activations
    Output: r_and2 ∈ R^(B × n_rules_l2a) — L2a compound conjunction rules
    """

    def __init__(
        self,
        n_rules_l1: int = _N_RULES_L1,
        n_rules_l2a: int = _N_RULES_L2A,
        init_range: float = 0.5,
        grad_graft: bool = False,
        eps: float = _EPS,
    ):
        super().__init__()
        self.n_rules_l1 = int(n_rules_l1)
        self.n_rules_l2a = int(n_rules_l2a)
        self.grad_graft = bool(grad_graft)
        self.eps = float(eps)

        # Trọng số W ∈ [0, 1] trực tiếp
        self.W = nn.Parameter(float(init_range) * torch.rand(self.n_rules_l1, self.n_rules_l2a))

    @property
    def weight_logit(self) -> torch.Tensor:
        return torch.logit(self.W.data.clamp(1e-4, 1.0 - 1e-4))

    def _effective_weights(self) -> torch.Tensor:
        return self.W.clamp(0.0, 1.0)

    def clip(self):
        self.W.data.clamp_(0.0, 1.0)

    def l1_norm(self) -> torch.Tensor:
        return torch.sum(self._effective_weights())

    def sparsity_loss(self) -> torch.Tensor:
        return self._effective_weights().mean()

    def continuous_forward(self, r_and: torch.Tensor) -> torch.Tensor:
        w = self._effective_weights()                       # [n_rules_l1, n_rules_l2a]
        X = 1.0 - (1.0 - r_and).unsqueeze(-1) * w           # [B, n_rules_l1, n_rules_l2a]
        return Product.apply(X)                             # [B, n_rules_l2a]

    @torch.no_grad()
    def binarized_forward(self, r_and: torch.Tensor) -> torch.Tensor:
        Wb = (self._effective_weights() > 0.5).float()
        return torch.prod(1.0 - (1.0 - r_and).unsqueeze(-1) * Wb, dim=1)

    def forward(self, r_and: torch.Tensor, continuous: bool = None) -> torch.Tensor:
        res_tilde = self.continuous_forward(r_and)
        use_cont = continuous if continuous is not None else (not self.grad_graft)
        if use_cont:
            return res_tilde
        res_bar = self.binarized_forward(r_and)
        return GradGraft.apply(res_bar, res_tilde)

    def extra_repr(self) -> str:
        return f"n_rules_l1={self.n_rules_l1}, n_rules_l2a={self.n_rules_l2a}, grad_graft={self.grad_graft}"





In [ ]:
# 3. DisjunctionLayer (L2b OR1)
"""DisjunctionLayer: Tầng OR rules cho CRL pipeline (L2b).

Module này thực hiện tuyển (disjunction) của các AND rules từ L1:
  r_and ∈ R^(B × 256)  →  r_or1 ∈ R^(B × 128)

Ý nghĩa: Mô tả các hành động có nhiều cách thực hiện khác nhau (alternative patterns).
Ví dụ: (arm_raise ∧ hand_lift) ∨ (hand_grasp ∧ arm_retract)

Công thức Probabilistic OR (De Morgan + log-space):
  r_or1_k = 1 - exp( Σ_j w_jk · log(1 - r_and_j + ε) )
           = 1 - ∏_j (1 - r_and_j + ε)^w_jk

  Trọng số w_jk = sigmoid(θ_jk) ∈ [0, 1]

Lưu ý:
  - Nếu tất cả w ≈ 0: r_or1 ≈ 0 (không kích hoạt)
  - Nếu ít nhất một AND rule j mạnh (r_and_j ≈ 1) và w_jk lớn:
    log(1 - r_and_j) → -∞  → r_or1_k → 1.0
"""

from typing import Optional

import torch
import torch.nn as nn

from .conjunction_layer import GradGraft, Product, _EPS


class DisjunctionLayer(nn.Module):
    """L2b OR1 layer: Tuyển của các AND rules (Alternative Action Patterns).

    Cải tiến theo 6 cơ chế REASON:
      1. W ∈ [0, 1] trực tiếp (khởi tạo 0.5 * rand, clip sau mỗi optimizer step)
      2. Łukasiewicz Product T-norm: 1 - Product.apply(1 - r_and * W)
      3. GradGraft: forward binary, backward continuous
      4. L1 Sparsity Penalty: l1_norm() và sparsity_loss()

    Input:  r_and  ∈ R^(B × n_rules_l1)  — L1 conjunction rule activations
    Output: r_or1  ∈ R^(B × n_rules_l2b) — L2b disjunction rule activations
    """

    def __init__(
        self,
        n_rules_l1: int = 256,
        n_rules_l2b: int = 128,
        init_range: float = 0.5,
        grad_graft: bool = False,
        eps: float = _EPS,
    ):
        super().__init__()
        self.n_rules_l1 = int(n_rules_l1)
        self.n_rules_l2b = int(n_rules_l2b)
        self.grad_graft = bool(grad_graft)
        self.eps = float(eps)

        # Trọng số W ∈ [0, 1] trực tiếp
        self.W = nn.Parameter(float(init_range) * torch.rand(self.n_rules_l1, self.n_rules_l2b))

    @property
    def weight_logit(self) -> torch.Tensor:
        """Hỗ trợ tương thích ngược cho code cũ truy cập weight_logit."""
        return torch.logit(self.W.data.clamp(1e-4, 1.0 - 1e-4))

    def _effective_weights(self) -> torch.Tensor:
        """Trọng số thực tế W ∈ [0, 1]."""
        return self.W.clamp(0.0, 1.0)

    def clip(self):
        """Clip trọng số vào đoạn [0, 1] sau optimizer step (REASON Mechanism 3)."""
        self.W.data.clamp_(0.0, 1.0)

    def l1_norm(self) -> torch.Tensor:
        """Tổng L1 norm cho sparsity penalty (REASON Mechanism 5)."""
        return torch.sum(self._effective_weights())

    def sparsity_loss(self) -> torch.Tensor:
        """Mean sparsity loss."""
        return self._effective_weights().mean()

    def continuous_forward(self, r_and: torch.Tensor) -> torch.Tensor:
        """Łukasiewicz Product T-norm tuyển liên tục: 1 - Product(1 - r_and * W)."""
        w = self._effective_weights()                       # [n_rules_l1, n_rules_l2b]
        X = 1.0 - r_and.unsqueeze(-1) * w                   # [B, n_rules_l1, n_rules_l2b]
        return 1.0 - Product.apply(X)                       # [B, n_rules_l2b]

    @torch.no_grad()
    def binarized_forward(self, r_and: torch.Tensor) -> torch.Tensor:
        """Binarized rule execution với W > 0.5."""
        Wb = (self._effective_weights() > 0.5).float()
        return 1.0 - torch.prod(1.0 - r_and.unsqueeze(-1) * Wb, dim=1)

    def forward(self, r_and: torch.Tensor, continuous: Optional[bool] = None) -> torch.Tensor:
        """Forward pass.

        Args:
            r_and: L1 conjunction activations [B, n_rules_l1]
            continuous: Nếu True, dùng continuous forward. Nếu None, dùng not self.grad_graft.
        """
        res_tilde = self.continuous_forward(r_and)
        use_cont = continuous if continuous is not None else (not self.grad_graft)
        if use_cont:
            return res_tilde
        res_bar = self.binarized_forward(r_and)
        return GradGraft.apply(res_bar, res_tilde)

    def extra_repr(self) -> str:
        return f"n_rules_l1={self.n_rules_l1}, n_rules_l2b={self.n_rules_l2b}, grad_graft={self.grad_graft}"





In [ ]:
# 4. CRLReasoner (1-Pass Shared Reasoner)
"""CRLReasoner: Concept Reasoning Logic Reasoner.

Module này gói gọn toàn bộ pipeline suy luận logic 1 lượt:
  z^t ∈ R^(B × 72)
    → LiteralLayer → l^t ∈ R^(B × 144)
    → ConjunctionLayer (L1 AND, Part-grouped) → r_and ∈ R^(B × 256)
    → [ConjunctionLayer2 (L2a AND2) ‖ DisjunctionLayer (L2b OR1)]
    → r_and2 ∈ R^(B × 128), r_or1 ∈ R^(B × 128)
    → ActionHead [Linear: 256 → 120] → y^t ∈ R^(B × 120)

    Logic Evidence (cho ConceptUpdateCell):
      r^t = [r_and; r_and2; r_or1] ∈ R^(B × 512)

Trọng số được chia sẻ (shared) giữa các vòng lặp trong IterativeLogicReasoner.

Thiết kế:
  - ActionHead KHÔNG nhận feature bypass từ backbone
    (đảm bảo tính giải thích tuyệt đối — strict logic mode)
  - Logic Evidence kết hợp cả ba tầng: L1 (body-part features),
    L2a (coordinated patterns), L2b (alternative patterns)
"""

from typing import Optional, Tuple

import torch
import torch.nn as nn

# relative import in notebook scope
# relative import in notebook scope
# relative import in notebook scope

# ─── Kích thước mặc định ────────────────────────────────────────────────────
_N_CONCEPTS  = 72   # số concepts (72 sau khi loại 2 null concepts)
_N_ACTIONS   = 120  # số lớp hành động NTU RGB+D 120
_R1          = 256  # L1 AND rules
_R2A         = 128  # L2a AND2 rules
_R2B         = 128  # L2b OR1 rules
_LOGIC_DIM   = _R1 + _R2A + _R2B  # = 512


class CRLReasoner(nn.Module):
    """Concept Reasoning Logic Reasoner (1-pass).

    Được dùng với shared weights trong IterativeLogicReasoner:
    mỗi vòng lặp gọi cùng 1 instance CRLReasoner này.

    Args:
        n_concepts:  Số concepts (mặc định 72).
        n_actions:   Số lớp hành động (mặc định 120).
        r1:          Số L1 AND rules (mặc định 256).
        r2a:         Số L2a AND2 rules (mặc định 128).
        r2b:         Số L2b OR1 rules (mặc định 128).
        literal_mode: 'soft' hoặc 'ste' (có thể thay đổi runtime).
        temperature: Nhiệt độ STE cho LiteralLayer.
    """

    def __init__(
        self,
        n_concepts:   int   = _N_CONCEPTS,
        n_actions:    int   = _N_ACTIONS,
        r1:           int   = _R1,
        r2a:          int   = _R2A,
        r2b:          int   = _R2B,
        literal_mode: str   = "ste",
        temperature:  float = 2.0,
    ):
        super().__init__()
        self.n_concepts  = int(n_concepts)
        self.n_actions   = int(n_actions)
        self.r1  = int(r1)
        self.r2a = int(r2a)
        self.r2b = int(r2b)
        self.logic_dim = r1 + r2a + r2b  # = 512 mặc định

        # ── Các sub-module ──────────────────────────────────────────────────
        self.literal_layer = LiteralLayer(
            n_concepts=n_concepts,
            mode=literal_mode,
            temperature=temperature,
        )

        self.l1_and = ConjunctionLayer(
            n_literals=2 * n_concepts,  # 144
            n_rules=r1,                 # 256
            part_grouped=False,         # REASON: Full-body connections
        )

        self.l2a_and2 = ConjunctionLayer2(
            n_rules_l1=r1,    # 256
            n_rules_l2a=r2a,  # 128
        )

        self.l2b_or1 = DisjunctionLayer(
            n_rules_l1=r1,    # 256
            n_rules_l2b=r2b,  # 128
        )

        # REASON Mechanism 6: ActionHead nhận r_evidence = [r_and1; r_and2; r_or1] (512 chiều)
        # Bao gồm skip-connection trực tiếp từ L1 primitive rules
        self.action_head = nn.Linear(self.logic_dim, n_actions)  # 512 → n_actions

    # ── Properties để điều chỉnh LiteralLayer & Logic Layers ─────────────────
    @property
    def literal_mode(self) -> str:
        return self.literal_layer.mode

    @literal_mode.setter
    def literal_mode(self, value: str):
        self.literal_layer.mode = value

    @property
    def temperature(self) -> float:
        return self.literal_layer.temperature

    @temperature.setter
    def temperature(self, value: float):
        self.literal_layer.temperature = float(value)

    @property
    def grad_graft(self) -> bool:
        return self.l1_and.grad_graft

    @grad_graft.setter
    def grad_graft(self, val: bool):
        self.l1_and.grad_graft = bool(val)
        self.l2a_and2.grad_graft = bool(val)
        self.l2b_or1.grad_graft = bool(val)

    # ── Forward ──────────────────────────────────────────────────────────────
    def forward(
        self,
        z: torch.Tensor,
        continuous: Optional[bool] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Suy luận logic 1 lượt.

        Args:
            z: Concept logits [B, n_concepts] (LUÔN là số thực liên tục)
            continuous: True nếu ép buộc continuous forward, False nếu ép binarized, None = theo layer.grad_graft

        Returns:
            r_evidence: Logic Evidence [B, logic_dim] (= [r_and; r_and2; r_or1], 512 chiều)
            y:          Action logits  [B, n_actions]
        """
        # Bước 1: Tạo literals qua RMSNorm + ScaledSoftsign
        l = self.literal_layer(z)              # [B, 2 * n_concepts]

        # Bước 2: L1 AND rules (Full-body, Łukasiewicz Product)
        r_and = self.l1_and(l, continuous=continuous)           # [B, 256]

        # Bước 3: L2 song song (L2a AND2 ∥ L2b OR1)
        r_and2 = self.l2a_and2(r_and, continuous=continuous)    # [B, 128]
        r_or1  = self.l2b_or1(r_and, continuous=continuous)     # [B, 128]

        # Bước 4: Logic Evidence 512 chiều: [r_and1 (256); r_and2 (128); r_or1 (128)]
        r_evidence = torch.cat([r_and, r_and2, r_or1], dim=-1) # [B, 512]

        # Bước 5: ActionHead nhận toàn bộ 512 chiều (strict logic mode + L1 skip connection)
        y = self.action_head(r_evidence)                        # [B, n_actions]

        return r_evidence, y

    # ── Logic weight clipping & Sparsity helpers ────────────────────────────
    def clip_weights(self):
        """Clip trọng số logic W ∈ [0, 1] sau optimizer step (REASON Mechanism 3)."""
        self.l1_and.clip()
        self.l2a_and2.clip()
        self.l2b_or1.clip()

    def l1_norm(self) -> torch.Tensor:
        """Tổng L1 norm của toàn bộ logic layers (REASON Mechanism 5)."""
        return self.l1_and.l1_norm() + self.l2a_and2.l1_norm() + self.l2b_or1.l1_norm()

    def rule_sparsity_loss(self) -> torch.Tensor:
        """Rule Sparsity Loss (mean weight loss)."""
        return (
            self.l1_and.sparsity_loss()
            + self.l2a_and2.sparsity_loss()
            + self.l2b_or1.sparsity_loss()
        ) / 3.0

    def sparsity_loss(self) -> torch.Tensor:
        """Alias cho rule_sparsity_loss."""
        return self.rule_sparsity_loss()

    def extra_repr(self) -> str:
        return (
            f"n_concepts={self.n_concepts}, n_actions={self.n_actions}, "
            f"r1={self.r1}, r2a={self.r2a}, r2b={self.r2b}, "
            f"logic_dim={self.logic_dim}"
        )





## 🔄 Block 5: ConceptUpdateCell, IterativeLogicReasoner & CLIRSkateFormer


In [ ]:
# CLIR Main Components
"""CLIR-SkateFormer: Concept Logic Iterative Refinement SkateFormer.

Kiến trúc chính:
  1. SkateFormer Backbone: Trích xuất feature map → concept logits z^0
  2. IterativeLogicReasoner (K=2 loops):
     - Mỗi vòng t: CRLReasoner(z^t) → (r^t, y^t) → ConceptUpdateCell → z^{t+1}
     - Final pass: CRLReasoner(z^K) → y^K (FINAL prediction)
  3. Loss System (Staged Training):
     - Phase 1 (Epoch 1-30):  L_act^K + L_sup (no L_reg)
     - Phase 2 (Epoch 31+):   L_act^K + L_sup + L_reg

Điểm đột phá:
  - Logic Evidence (r^t) từ DNF rules dẫn dắt concept refinement (top-down)
  - ActionHead KHÔNG nhận feature bypass (strict logic interpretability)
  - Binarization chỉ ở LiteralLayer, z^t luôn là logit liên tục
"""

from __future__ import annotations

from collections import OrderedDict
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union

import torch
import torch.nn as nn
import torch.nn.functional as F

# relative import in notebook scope
# relative import in notebook scope

# ─── Hằng số mặc định ────────────────────────────────────────────────────────

_N_CONCEPTS_SPATIAL  = 52   # concepts không gian sau khi loại bỏ null concept
_N_CONCEPTS_TEMPORAL = 20   # concepts thời gian sau khi loại bỏ null concept
_N_CONCEPTS          = 72   # = 52 + 20 (khớp với feeder_ntu.py remove_null_concept=True)
_N_ACTIONS           = 120  # NTU RGB+D 120
_LOGIC_DIM           = 512  # = r1(256) + r2a(128) + r2b(128)


# ─────────────────────────────────────────────────────────────────────────────
# ConceptUpdateCell
# ─────────────────────────────────────────────────────────────────────────────

class ConceptUpdateCell(nn.Module):
    """Logic-Guided Concept Update Cell (Top-Down Feedback).

    Thay vì KG tĩnh, dùng Logic Evidence r^t (từ DNF rules) để tinh chỉnh z^t.

    Cơ chế:
        z^t (concept logits) ──┐
                               ├──► Concat [z^t, Proj(r^t)] ──► MLP ──► Δz^t
        r^t (logic evidence) ──┘      (72 + 72 = 144)

        z^{t+1} = z^t + β^t · Δz^t

    Args:
        num_concepts: Số concepts (mặc định 72).
        logic_dim:    Chiều của Logic Evidence r^t (mặc định 512 = R1+R2a+R2b).
        hidden_dim:   Chiều ẩn của MLP (mặc định 256).
        dropout:      Dropout rate (mặc định 0.1).
    """

    def __init__(
        self,
        num_concepts: int = _N_CONCEPTS,
        logic_dim:    int = _LOGIC_DIM,
        hidden_dim:   int = 256,
        dropout:      float = 0.1,
    ):
        super().__init__()
        self.num_concepts = int(num_concepts)
        self.logic_dim    = int(logic_dim)

        # Project logic evidence từ 512 → 72
        self.logic_proj = nn.Sequential(
            nn.Linear(logic_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, num_concepts),   # 256 → 72
        )

        # Update MLP: [z^t (72), r_proj (72)] → Δz^t (72)
        self.update_mlp = nn.Sequential(
            nn.Linear(2 * num_concepts, hidden_dim),   # 144 → 256
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_concepts),        # 256 → 72
        )

    def forward(self, z_t: torch.Tensor, r_t: torch.Tensor) -> torch.Tensor:
        """Tính residual update Δz^t.

        Args:
            z_t: Concept logits hiện tại [B, num_concepts]  (liên tục)
            r_t: Logic Evidence từ CRLReasoner [B, logic_dim]

        Returns:
            delta_z: [B, num_concepts]  (residual, chưa nhân β)
        """
        r_proj = self.logic_proj(r_t)                    # [B, 72]
        cell_input = torch.cat([z_t, r_proj], dim=-1)    # [B, 144]
        return self.update_mlp(cell_input)                # [B, 72]


# ─────────────────────────────────────────────────────────────────────────────
# IterativeLogicReasoner
# ─────────────────────────────────────────────────────────────────────────────

class IterativeLogicReasoner(nn.Module):
    """Vòng lặp suy luận lặp K=2 với Shared CRLReasoner.

    Luồng dữ liệu (K=2):
      z^0 → CRLReasoner → (r^0, y^0) → ConceptUpdateCell → Δz^0 → z^1
      z^1 → CRLReasoner → (r^1, y^1) → ConceptUpdateCell → Δz^1 → z^2
      z^2 → CRLReasoner [FINAL] → y^2  ← FINAL PREDICTION

    Trọng số CRLReasoner và ConceptUpdateCell được chia sẻ qua tất cả vòng.

    Args:
        num_concepts:   Số concepts (72).
        num_actions:    Số lớp hành động (120).
        num_loops:      Số vòng lặp tinh chỉnh K (mặc định 2).
        r1, r2a, r2b:   Kích thước rule banks.
        beta_max:       Giá trị tối đa của β (mặc định 0.5 ở Phase 2).
        beta_init:      Giá trị khởi tạo của β (mặc định 0.1 ở Phase 1).
        literal_mode:   'soft' hoặc 'ste'.
        temperature:    Nhiệt độ STE.
        hidden_dim:     Chiều ẩn của ConceptUpdateCell.
        dropout:        Dropout rate.
    """

    def __init__(
        self,
        num_concepts: int   = _N_CONCEPTS,
        num_actions:  int   = _N_ACTIONS,
        num_loops:    int   = 2,
        r1:           int   = 256,
        r2a:          int   = 128,
        r2b:          int   = 128,
        beta_max:     float = 0.5,
        beta_init:    float = 0.1,
        literal_mode: str   = "ste",
        temperature:  float = 2.0,
        hidden_dim:   int   = 256,
        dropout:      float = 0.1,
        random_feedback: bool = False,
    ):
        super().__init__()
        self.num_concepts = int(num_concepts)
        self.num_actions  = int(num_actions)
        self.num_loops    = max(0, int(num_loops))  # Cho phép num_loops=0 (A2: True No-Loop Baseline)
        self.beta_max     = float(beta_max)
        self.random_feedback = bool(random_feedback)
        logic_dim = r1 + r2a + r2b

        # ── Shared CRLReasoner (dùng lại qua tất cả vòng lặp) ──────────────
        self.crl = CRLReasoner(
            n_concepts=num_concepts,
            n_actions=num_actions,
            r1=r1,
            r2a=r2a,
            r2b=r2b,
            literal_mode=literal_mode,
            temperature=temperature,
        )

        # ── Shared ConceptUpdateCell ─────────────────────────────────────────
        self.update_cell = ConceptUpdateCell(
            num_concepts=num_concepts,
            logic_dim=logic_dim,
            hidden_dim=hidden_dim,
            dropout=dropout,
        )

        # ── Learnable β per loop ─────────────────────────────────────────────
        # β^t = sigmoid(raw_β^t) × β_max
        # Xử lý an toàn khi num_loops = 0 (A2: True No-Loop) hoặc beta_max <= 0
        if self.num_loops == 0:
            self.raw_beta = nn.Parameter(torch.empty(0, dtype=torch.float32))
        elif self.beta_max <= 0.0:
            raw_init = torch.tensor(-10.0, dtype=torch.float32)
            self.raw_beta = nn.Parameter(
                raw_init.unsqueeze(0).expand(self.num_loops).clone()
            )
        else:
            ratio = min(max(beta_init / self.beta_max, 1e-4), 1.0 - 1e-4)
            raw_init = torch.logit(torch.tensor(ratio, dtype=torch.float32))
            self.raw_beta = nn.Parameter(
                raw_init.unsqueeze(0).expand(self.num_loops).clone()
            )

    # ── Properties để điều chỉnh LiteralLayer ────────────────────────────────
    @property
    def literal_mode(self) -> str:
        return self.crl.literal_mode

    @literal_mode.setter
    def literal_mode(self, value: str):
        self.crl.literal_mode = value

    @property
    def temperature(self) -> float:
        return self.crl.temperature

    @temperature.setter
    def temperature(self, value: float):
        self.crl.temperature = float(value)

    def get_betas(self) -> torch.Tensor:
        """Lấy β^t cho mỗi vòng lặp t ∈ [0, K-1] — shape [K]."""
        if self.num_loops == 0:
            return self.raw_beta
        if self.beta_max <= 0.0:
            return torch.zeros_like(self.raw_beta)
        return torch.sigmoid(self.raw_beta) * self.beta_max

    # ── Forward ──────────────────────────────────────────────────────────────
    def forward(
        self,
        z0: torch.Tensor,
        return_intermediates: bool = True,
    ) -> Dict:
        """Thực hiện K vòng lặp suy luận + 1 final pass.

        Args:
            z0:                   Initial concept logits [B, num_concepts]
            return_intermediates: Có lưu các y^t và z^t trung gian không

        Returns:
            dict với keys:
              - y_final:       Final action logits y^K [B, num_actions]
              - z_final:       Final concept logits z^K [B, num_concepts]
              - z_all:         List [z^0, z^1, ..., z^K]
              - y_all:         List [y^0, y^1, ..., y^{K-1}]  (trung gian)
              - betas:         [K] tensor
        """
        betas = self.get_betas()   # [K]
        z_t = z0
        z_all = [z0] if return_intermediates else []
        y_all = []

        # ── K vòng lặp tinh chỉnh ────────────────────────────────────────────
        for t in range(self.num_loops):
            # Suy luận logic: z^t → (r^t, y^t)
            r_t, y_t = self.crl(z_t)
            y_all.append(y_t)

            # A5 Ablation: Random Logic Feedback (thay r_t bằng tensor ngẫu nhiên)
            if self.random_feedback:
                feedback_r = torch.rand_like(r_t)
            else:
                feedback_r = r_t

            # Tính residual: Δz^t
            delta_t = self.update_cell(z_t, feedback_r)

            # Cập nhật concept logits: z^{t+1} = z^t + β^t · Δz^t
            z_t = z_t + betas[t] * delta_t

            if return_intermediates:
                z_all.append(z_t)

        # ── Final pass (không cập nhật thêm) ─────────────────────────────────
        _, y_final = self.crl(z_t)

        return {
            "y_final": y_final,
            "z_final": z_t,
            "z_all":   z_all,
            "y_all":   y_all,   # [y^0, ..., y^{K-1}]
            "betas":   betas,
        }

    def clip_weights(self):
        """Clip logic layer weights W ∈ [0, 1] sau optimizer step (REASON Mechanism 3)."""
        self.crl.clip_weights()

    def l1_norm(self) -> torch.Tensor:
        """Tổng L1 norm của logic layers (REASON Mechanism 5)."""
        return self.crl.l1_norm()

    def sparsity_loss(self) -> torch.Tensor:
        """Sparsity loss từ CRLReasoner."""
        return self.crl.sparsity_loss()

    def rule_sparsity_loss(self) -> torch.Tensor:
        """Ủy quyền cho CRLReasoner."""
        return self.crl.rule_sparsity_loss()

    @property
    def grad_graft(self) -> bool:
        return self.crl.grad_graft

    @grad_graft.setter
    def grad_graft(self, val: bool):
        self.crl.grad_graft = bool(val)



# ─────────────────────────────────────────────────────────────────────────────
# CLIRSkateFormer
# ─────────────────────────────────────────────────────────────────────────────

class CLIRSkateFormer(nn.Module):
    """CLIR-SkateFormer: Concept Logic Iterative Refinement for Skeleton AR.

    Tích hợp SkateFormer backbone với IterativeLogicReasoner để thực hiện
    tinh chỉnh concept lặp K=2 vòng được dẫn dắt bởi logic evidence.

    Luồng dữ liệu:
        X ∈ R^(B × C × T × V × M)
          → SkateFormer.forward_features_from_input(X)
          → feature_map ∈ R^(B × D × T' × V')
          → spatial_pool_score, temporal_pool_score (attention)
          → z^0 = [z_s (52); z_t (20)] ∈ R^(B × 72)
          → IterativeLogicReasoner (K=2)
          → y^K ∈ R^(B × 120)  [FINAL PREDICTION]

    Args:
        num_classes:         Số classes (120).
        num_concepts:        Tổng số concepts = spatial + temporal (72).
        spatial_concepts:    Số spatial concepts (52).
        temporal_concepts:   Số temporal concepts (20).
        num_people:          Số người trong skeleton (2).
        num_points:          Số keypoints (24).
        num_frames:          Số frames (64).
        backbone_weights:    Path đến SkateFormer pretrained weights.
        concept_names:       Danh sách tên concepts.
        num_loops:           Số vòng lặp K (2).
        r1, r2a, r2b:        Kích thước rule banks (256, 128, 128).
        beta_max:            Giá trị tối đa của β (0.5).
        beta_init:           Khởi tạo của β (0.1).
        literal_mode:        'ste' (mặc định Pure STE) hoặc 'soft'.
        temperature:         Nhiệt độ STE (mặc định 2.0, annealing dần tới 10.0).
        hidden_dim:          Chiều ẩn ConceptUpdateCell (256).
        dropout:             Dropout rate (0.1).
        lambda_sup:          Trọng số L_sup (1.0 cố định).
        lambda_reg:          Trọng số L_reg (mặc định 0.005 cho rule sparsity + loop stability).
        concept_negative_weight: Trọng số negative trong Asymmetric BCE (0.15).
        asymmetric_gamma_neg:    γ- cho Asymmetric BCE (2.0).
        return_aux:          Có trả về thông tin phụ không (True khi train).
    """

    def __init__(
        self,
        num_classes:              int   = _N_ACTIONS,
        num_concepts:             int   = _N_CONCEPTS,
        spatial_concepts:         int   = _N_CONCEPTS_SPATIAL,
        temporal_concepts:        int   = _N_CONCEPTS_TEMPORAL,
        num_people:               int   = 2,
        num_points:               int   = 24,
        num_frames:               int   = 64,
        backbone_weights:         Optional[str]       = None,
        concept_names:            Optional[List[str]] = None,
        # Reasoner
        num_loops:    int   = 2,
        r1:           int   = 256,
        r2a:          int   = 128,
        r2b:          int   = 128,
        beta_max:     float = 0.5,
        beta_init:    float = 0.1,
        literal_mode: str   = "ste",
        temperature:  float = 2.0,
        hidden_dim:   int   = 256,
        dropout:      float = 0.1,
        # Loss weights
        lambda_sup:              float = 1.0,
        lambda_reg:              float = 0.005,
        concept_negative_weight: float = 0.15,
        concept_positive_weight: float = 1.0,
        asymmetric_gamma_neg:    float = 2.0,
        asymmetric_clip:         float = 0.05,
        return_aux:              bool  = True,
        # Ablations
        use_feature_bypass:      bool  = False,  # A6: CLIR-Bypass
        random_feedback:         bool  = False,  # A5: Random Logic Feedback
        **kwargs,
    ):
        super().__init__()

        if spatial_concepts + temporal_concepts != num_concepts:
            raise ValueError(
                f"spatial_concepts({spatial_concepts}) + temporal_concepts({temporal_concepts})"
                f" != num_concepts({num_concepts})"
            )

        self.num_classes         = int(num_classes)
        self.num_concepts        = int(num_concepts)
        self.spatial_concepts    = int(spatial_concepts)
        self.temporal_concepts   = int(temporal_concepts)
        self.num_loops           = max(0, int(num_loops))
        self.lambda_sup          = float(lambda_sup)
        self.lambda_reg          = float(lambda_reg)
        self.concept_neg_weight  = float(concept_negative_weight)
        self.concept_pos_weight  = float(concept_positive_weight)
        self.gamma_neg           = float(asymmetric_gamma_neg)
        self.asymmetric_clip     = float(asymmetric_clip)
        self.return_aux          = bool(return_aux)
        self.use_feature_bypass  = bool(use_feature_bypass)
        self.accepts_concepts    = True   # flag cho training loop

        # ── Backbone ─────────────────────────────────────────────────────────
        self.backbone = SkateFormer_(
            num_classes=num_classes,
            num_people=num_people,
            num_points=num_points,
            num_frames=num_frames,
            **kwargs,
        )
        self._load_backbone_weights(backbone_weights)
        feature_dim = self.backbone.head.in_features

        # A6: CLIR-Bypass Head (kết nối trực tiếp từ feature map vào ActionHead)
        if self.use_feature_bypass:
            self.bypass_head = nn.Linear(feature_dim, self.num_classes)
        else:
            self.bypass_head = None

        # ── Concept Heads (Spatial + Temporal) ───────────────────────────────
        self.spatial_pool_score = nn.Conv1d(feature_dim, 1, kernel_size=1)
        self.temporal_pool_score = nn.Conv1d(feature_dim, 1, kernel_size=1)

        self.spatial_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, self.spatial_concepts),
        )
        self.temporal_head = nn.Sequential(
            nn.Linear(feature_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, self.temporal_concepts),
        )

        # concept_names
        if concept_names is None:
            concept_names = [f"concept_{i}" for i in range(self.num_concepts)]
        self.concept_names = list(concept_names)

        # ── IterativeLogicReasoner ────────────────────────────────────────────
        self.reasoner = IterativeLogicReasoner(
            num_concepts=self.num_concepts,
            num_actions=self.num_classes,
            num_loops=self.num_loops,
            r1=r1,
            r2a=r2a,
            r2b=r2b,
            beta_max=beta_max,
            beta_init=beta_init,
            literal_mode=literal_mode,
            temperature=temperature,
            hidden_dim=hidden_dim,
            dropout=dropout,
            random_feedback=random_feedback,
        )

    # ── Backbone weight loading ───────────────────────────────────────────────

    def _load_backbone_weights(self, checkpoint_path: Optional[str]):
        if not checkpoint_path:
            return
        path = Path(checkpoint_path)
        if not path.exists():
            print(f"[Warning] Backbone checkpoint not found: {path}. Initializing randomly.")
            return
        checkpoint = torch.load(str(path), map_location="cpu")
        if isinstance(checkpoint, dict):
            for key in ("state_dict", "model"):
                if key in checkpoint and isinstance(checkpoint[key], dict):
                    checkpoint = checkpoint[key]
                    break
        if not isinstance(checkpoint, dict):
            raise TypeError("Unsupported SkateFormer checkpoint format")

        backbone_state = self.backbone.state_dict()
        mapped = OrderedDict()
        for key, value in checkpoint.items():
            clean_key = key.replace("module.", "", 1)
            if clean_key.startswith("backbone."):
                clean_key = clean_key[len("backbone."):]
            if clean_key in backbone_state:
                if backbone_state[clean_key].shape == value.shape:
                    mapped[clean_key] = value
                else:
                    print(
                        f"[Warning] Skipping {clean_key} due to shape mismatch: "
                        f"checkpoint {value.shape} vs model {backbone_state[clean_key].shape}"
                    )
        missing, unexpected = self.backbone.load_state_dict(mapped, strict=False)
        print(
            f"Loaded {len(mapped)} SkateFormer backbone tensors from {path}; "
            f"missing={len(missing)}, unexpected={len(unexpected)}"
        )

    # ── Pooling ───────────────────────────────────────────────────────────────

    def _pool_features(
        self, feature_map: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """Attention pooling dọc theo spatial và temporal dimensions.

        Args:
            feature_map: [B, D, T', V']

        Returns:
            spatial_feat:  [B, D]
            temporal_feat: [B, D]
        """
        # Spatial: pool over T' dimension → [B, D, V']
        spatial_tokens = feature_map.mean(dim=2)                              # [B, D, V']
        spatial_attn   = torch.softmax(self.spatial_pool_score(spatial_tokens), dim=-1)  # [B, 1, V']
        spatial_feat   = (spatial_tokens * spatial_attn).sum(dim=-1)         # [B, D]

        # Temporal: pool over V' dimension → [B, D, T']
        temporal_tokens = feature_map.mean(dim=3)                              # [B, D, T']
        temporal_attn   = torch.softmax(self.temporal_pool_score(temporal_tokens), dim=-1)
        temporal_feat   = (temporal_tokens * temporal_attn).sum(dim=-1)       # [B, D]

        return spatial_feat, temporal_feat

    # ── Loss Functions ────────────────────────────────────────────────────────

    def _asymmetric_bce(
        self,
        logits:    torch.Tensor,
        targets:   torch.Tensor,
        mask:      Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """Asymmetric BCE loss (Mục 8.1 — CLIR plan, chuẩn ASL Alibaba ICCV 2021).

        L_c(z, ĉ) = Σ r_ic [ĉ_ic · softplus(-z_ic)
                             + (1-ĉ_ic) · clamp(σ(z_ic) - m, 0)^γ- · softplus(z_ic)] / Σ r_ic

        Với r_ic: positive_weight=1.0, negative_weight=0.15, m=0.05 (margin clipping), γ-=2.0.

        Args:
            logits:  [B, n_concepts]
            targets: [B, n_concepts] ∈ {0, 1}  (soft labels được cho phép)
            mask:    [B, n_concepts] binary mask (None = dùng tất cả)

        Returns:
            scalar loss
        """
        targets = targets.float().clamp(0.0, 1.0)
        probs = torch.sigmoid(logits)

        # Margin clipping m=0.05: Triệt tiêu 100% loss và gradient của easy negatives (p <= 0.05)
        clip_m = getattr(self, "asymmetric_clip", 0.05)
        p_m = torch.clamp(probs - clip_m, min=0.0)

        pos_loss = F.softplus(-logits) * targets
        neg_loss = (
            F.softplus(logits)
            * p_m.pow(self.gamma_neg)
            * (1.0 - targets)
        )

        weights = (
            targets * self.concept_pos_weight
            + (1.0 - targets) * self.concept_neg_weight
        )
        if mask is not None:
            weights = weights * mask.float()

        loss = pos_loss + neg_loss
        denom = weights.sum().clamp_min(1.0)
        return (loss * weights).sum() / denom

    def compute_sup_loss(
        self,
        z0:              torch.Tensor,
        z_all:           List[torch.Tensor],
        y_all:           List[torch.Tensor],
        concept_targets: torch.Tensor,
        action_labels:   torch.Tensor,
        concept_mask:    Optional[torch.Tensor] = None,
    ) -> Dict[str, torch.Tensor]:
        """Tính L_sup = L_c^K + 0.5 · L_c^0 + 0.3 · (1/K) Σ L_act^t.

        Args:
            z0:              Initial concept logits [B, 72]
            z_all:           [z^0, z^1, ..., z^K]
            y_all:           [y^0, ..., y^{K-1}] (intermediate action logits)
            concept_targets: [B, 72]
            action_labels:   [B]
            concept_mask:    [B, 72] optional

        Returns:
            dict với: l_c_K, l_c_0, l_act_aux, l_sup
        """
        z_K = z_all[-1]   # Final concept logits

        # L_c^K: concept loss ở bước cuối
        l_c_K = self._asymmetric_bce(z_K, concept_targets, mask=concept_mask)

        # L_c^0: concept loss sớm để ép backbone học concept chuẩn
        l_c_0 = self._asymmetric_bce(z0, concept_targets, mask=concept_mask)

        # L_act^t: auxiliary action loss trên các bước trung gian
        l_act_aux = z0.new_zeros(())
        if len(y_all) > 0:
            for y_t in y_all:
                l_act_aux = l_act_aux + F.cross_entropy(y_t, action_labels)
            l_act_aux = l_act_aux / len(y_all)

        # L_sup = L_c^K + 0.5 · L_c^0 + 0.3 · L_act_aux
        l_sup = l_c_K + 0.5 * l_c_0 + 0.3 * l_act_aux

        return {
            "l_c_K":     l_c_K,
            "l_c_0":     l_c_0,
            "l_act_aux": l_act_aux,
            "l_sup":     l_sup,
        }

    def compute_reg_loss(
        self,
        z_all: List[torch.Tensor],
    ) -> Dict[str, torch.Tensor]:
        """Tính L_reg = L_sparse + L_stable.

        L_sparse: Ép thưa các trọng số luật
        L_stable: Giữ cho tinh chỉnh hội tụ, không dao động

        Args:
            z_all: [z^0, z^1, ..., z^K]

        Returns:
            dict với: l_sparse, l_stable, l_reg
        """
        # L_sparse: từ CRLReasoner
        l_sparse = self.reasoner.rule_sparsity_loss()

        # L_stable: Σ ||σ(z^{t+1}) - σ(z^t)||_1 / K
        l_stable = z_all[0].new_zeros(())
        K = len(z_all) - 1   # số bước update thực sự
        if K > 0:
            for t in range(K):
                diff = (torch.sigmoid(z_all[t + 1]) - torch.sigmoid(z_all[t])).abs()
                l_stable = l_stable + diff.mean()
            l_stable = l_stable / K

        l_reg = l_sparse + l_stable

        return {
            "l_sparse": l_sparse,
            "l_stable": l_stable,
            "l_reg":    l_reg,
        }

    # ── Forward ──────────────────────────────────────────────────────────────

    def forward(
        self,
        input:                  torch.Tensor,
        index_t:                Optional[torch.Tensor] = None,
        concept_targets:        Optional[torch.Tensor] = None,
        concept_target_mask:    Optional[torch.Tensor] = None,
        action_labels:          Optional[torch.Tensor] = None,
        return_trace:           bool = False,
    ) -> Union[torch.Tensor, Dict]:
        """Forward pass của CLIRSkateFormer.

        Args:
            input:               [B, C, T, V, M]
            index_t:             Temporal indices (cho SkateFormer)
            concept_targets:     [B, 72] nhãn concept (0/1)
            concept_target_mask: [B, 72] mask nhãn
            action_labels:       [B] nhãn action
            return_trace:        Có trả về intermediate states không

        Returns:
            Khi không train và không có labels: tensor y_final [B, n_actions]
            Khi train hoặc có labels: dict với loss và intermediate states
        """
        # ── Backbone feature extraction ───────────────────────────────────────
        feature_map = self.backbone.forward_features_from_input(input, index_t=index_t)
        # [B, D, T', V']

        spatial_feat, temporal_feat = self._pool_features(feature_map)

        # ── Concept logits z^0 ────────────────────────────────────────────────
        z_s = self.spatial_head(spatial_feat)      # [B, 52]
        z_t = self.temporal_head(temporal_feat)    # [B, 20]
        z0  = torch.cat([z_s, z_t], dim=-1)       # [B, 72]

        # ── Iterative Logic Reasoning ─────────────────────────────────────────
        reasoner_out = self.reasoner(z0, return_intermediates=True)
        y_final = reasoner_out["y_final"]     # [B, 120]
        z_final = reasoner_out["z_final"]     # [B, 72]
        z_all   = reasoner_out["z_all"]       # [z^0, ..., z^K]
        y_all   = list(reasoner_out["y_all"]) # [y^0, ..., y^{K-1}]
        betas   = reasoner_out["betas"]       # [K]

        # A6: CLIR-Bypass connection (thêm trực tiếp từ visual features vào ActionHead)
        if self.use_feature_bypass and self.bypass_head is not None:
            bypass_feat = 0.5 * (spatial_feat + temporal_feat)
            bypass_logits = self.bypass_head(bypass_feat)
            y_final = y_final + bypass_logits
            y_all = [y + bypass_logits for y in y_all]

        # ── Inference mode (không train) ──────────────────────────────────────
        if not self.return_aux and concept_targets is None and not self.training:
            return y_final

        # ── Tính Losses ───────────────────────────────────────────────────────
        output = {
            "action_logits":         y_final,
            "concept_logits":        z_final,
            "initial_concept_logits": z0,
            "spatial_logits":        z_s,
            "temporal_logits":       z_t,
            "all_action_logits":     y_all,
            "betas":                 betas,
        }

        if return_trace:
            output["z_all"] = z_all

        if concept_targets is not None and action_labels is not None:
            targets = concept_targets.float().to(z0.device)

            # L_sup
            sup_losses = self.compute_sup_loss(
                z0=z0,
                z_all=z_all,
                y_all=y_all,
                concept_targets=targets,
                action_labels=action_labels,
                concept_mask=concept_target_mask,
            )
            output.update(sup_losses)

            # L_reg (chỉ khi lambda_reg > 0, tức Phase 2)
            if self.lambda_reg > 0.0:
                reg_losses = self.compute_reg_loss(z_all)
                output.update(reg_losses)
                output["aux_loss"] = (
                    self.lambda_sup * sup_losses["l_sup"]
                    + self.lambda_reg * reg_losses["l_reg"]
                )
            else:
                output["l_reg"] = z0.new_zeros(())
                output["aux_loss"] = self.lambda_sup * sup_losses["l_sup"]

        elif concept_targets is not None:
            # Có concept targets nhưng không có action labels
            targets = concept_targets.float().to(z0.device)
            l_c_K = self._asymmetric_bce(z_final, targets, mask=concept_target_mask)
            l_c_0 = self._asymmetric_bce(z0, targets, mask=concept_target_mask)
            output["l_c_K"] = l_c_K
            output["l_c_0"] = l_c_0
            output["aux_loss"] = l_c_K + 0.5 * l_c_0

        return output

    # ── Convenience methods ───────────────────────────────────────────────────

    def set_phase(self, phase: int, temperature: Optional[float] = None):
        """Chuyển đổi training phase.

        phase=1: Concept warm-up (soft mode, beta_max=0.1, lambda_reg=0)
        phase=2: STE + Rule sparsification (ste mode, beta_max=0.5, lambda_reg=0.005)

        Args:
            phase:       1 hoặc 2.
            temperature: Ghi đè temperature (None = dùng mặc định).
        """
        if phase == 1:
            self.reasoner.literal_mode = "soft"
            if self.reasoner.num_loops > 0 and self.reasoner.beta_max > 0.0:
                self.reasoner.beta_max = 0.1
                with torch.no_grad():
                    # Hiệu chỉnh raw_beta sao cho beta ≈ 0.10 ở Phase 1 (thay vì tụt xuống 0.02)
                    target_ratio = 0.95
                    self.reasoner.raw_beta.fill_(float(torch.logit(torch.tensor(target_ratio))))
            self.lambda_reg = 0.0
            if temperature is not None:
                self.reasoner.temperature = temperature
        elif phase == 2:
            self.reasoner.literal_mode = "ste"
            if self.reasoner.num_loops > 0 and self.reasoner.beta_max > 0.0:
                old_beta_max = self.reasoner.beta_max
                new_beta_max = 0.5
                with torch.no_grad():
                    # Bảo toàn giá trị beta hiện tại từ Phase 1 (≈ 0.10), chuyển trơn sang beta_max = 0.5
                    current_beta = torch.sigmoid(self.reasoner.raw_beta) * old_beta_max
                    new_ratio = (current_beta / new_beta_max).clamp(1e-4, 1.0 - 1e-4)
                    self.reasoner.raw_beta.copy_(torch.logit(new_ratio))
                self.reasoner.beta_max = new_beta_max
            self.lambda_reg = 0.005
            if temperature is not None:
                self.reasoner.temperature = temperature
        else:
            raise ValueError(f"phase phải là 1 hoặc 2, nhận được: {phase}")
        print(
            f"[CLIRSkateFormer] Switched to Phase {phase}: "
            f"mode={self.reasoner.literal_mode!r}, "
            f"beta_max={self.reasoner.beta_max}, "
            f"lambda_reg={self.lambda_reg}, "
            f"temperature={self.reasoner.temperature}"
        )

    def clip_weights(self):
        """Clip logic layer weights W ∈ [0, 1] sau optimizer step (REASON Mechanism 3)."""
        self.reasoner.clip_weights()

    def l1_norm(self) -> torch.Tensor:
        """Tổng L1 norm của logic layers (REASON Mechanism 5)."""
        return self.reasoner.l1_norm()

    def sparsity_loss(self) -> torch.Tensor:
        """Sparsity loss từ reasoner."""
        return self.reasoner.sparsity_loss()



# ─────────────────────────────────────────────────────────────────────────────
# Factory functions
# ─────────────────────────────────────────────────────────────────────────────

def CLIR_SkateFormer(**kwargs) -> CLIRSkateFormer:
    """Factory function cho CLIRSkateFormer."""
    return CLIRSkateFormer(**kwargs)




## 📜 Block 6: RuleExtractor (DNF Rules & Fidelity Evaluation)


In [ ]:
# RuleExtractor
"""RuleExtractor: Trích xuất và đánh giá luật DNF sau khi huấn luyện.

Module này thực hiện post-hoc extraction của các luật DNF từ mô hình CLIR đã huấn luyện:
1. Binarize các trọng số theo ngưỡng tau:
   - w_and1 = (sigmoid(theta_and1) >= tau_and)   [L1 AND rules]
   - w_and2 = (sigmoid(theta_and2) >= tau_and2)  [L2a AND2 rules]
   - w_or1  = (sigmoid(theta_or1)  >= tau_or)    [L2b OR1 rules]
   - W_action: lọc các trọng số dương > tau_act (excitatory clauses)

2. Executable DNF Engine:
   - Đánh giá luật dạng boolean/nhị phân trên từng mẫu dữ liệu
   - Kiểm tra L1 conjunction rules (tất cả literal thành phần = 1)
   - Kiểm tra L2a compound rules (tất cả L1 thành phần = 1)
   - Kiểm tra L2b alternative rules (ít nhất 1 L1 thành phần = 1)
   - Tính toán điểm kích hoạt lớp từ các mệnh đề DNF dương

3. Chỉ số đánh giá theo kế hoạch (Mục 10 — CLIR plan):
   - Rule Coverage: Tỷ lệ mẫu được bao phủ bởi ít nhất 1 luật DNF
   - Rule Precision: Tỷ lệ mẫu thỏa mãn luật thực sự thuộc về class đó
   - Model-Rule Fidelity: Tỷ lệ tương đồng giữa dự đoán của mạng nơ-ron và luật logic tượng trưng
"""

from __future__ import annotations
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn

# relative import in notebook scope


_DEFAULT_CONCEPT_NAMES = [f"concept_{i}" for i in range(_DEFAULT_N_CONCEPTS)]


class RuleExtractor:
    """Trình trích xuất và thực thi luật DNF từ CRLReasoner đã huấn luyện.

    Args:
        reasoner:       Instance của CRLReasoner (sau khi load checkpoint).
        concept_names:  Danh sách tên concepts.
        action_names:   Danh sách tên actions.
        tau_and:        Ngưỡng binarize cho L1 AND weights (mặc định 0.5).
        tau_and2:       Ngưỡng binarize cho L2a AND2 weights (mặc định 0.5).
        tau_or:         Ngưỡng binarize cho L2b OR1 weights (mặc định 0.5).
        tau_act:        Ngưỡng trọng số dương ActionHead (mặc định 0.05).
        top_k_l2:       Số L2 rules tối đa cho mỗi action (mặc định 5).
        top_k_l1:       Số L1 rules tối đa trong mỗi L2 rule (mặc định 5).
        top_k_literals: Số literals tối đa trong mỗi L1 rule (mặc định 6).
    """

    def __init__(
        self,
        reasoner,
        concept_names:  Optional[List[str]] = None,
        action_names:   Optional[List[str]] = None,
        tau_and:        float = 0.5,
        tau_and2:       float = 0.5,
        tau_or:         float = 0.5,
        tau_act:        float = 0.05,
        top_k_l2:       int = 5,
        top_k_l1:       int = 5,
        top_k_literals: int = 6,
    ):
        self.reasoner       = reasoner
        self.n_concepts     = reasoner.n_concepts
        self.n_actions      = reasoner.n_actions
        self.concept_names  = concept_names or _DEFAULT_CONCEPT_NAMES[:self.n_concepts]
        self.action_names   = action_names  or [f"action_{i}" for i in range(self.n_actions)]
        self.tau_and        = float(tau_and)
        self.tau_and2       = float(tau_and2)
        self.tau_or         = float(tau_or)
        self.tau_act        = float(tau_act)
        self.top_k_l2       = int(top_k_l2)
        self.top_k_l1       = int(top_k_l1)
        self.top_k_literals = int(top_k_literals)

    # ─── Lấy trọng số binarized (giữ nguyên device hiện tại) ───────────────────

    # ─── Lấy trọng số binarized (giữ nguyên device hiện tại) ───────────────────

    @torch.no_grad()
    def _get_w_and1(self) -> torch.Tensor:
        """W_and1 binarized [n_literals, r1]."""
        if hasattr(self.reasoner.l1_and, "W"):
            w = self.reasoner.l1_and.W.data.clamp(0.0, 1.0)
        else:
            w = torch.sigmoid(self.reasoner.l1_and.weight_logit)
        if getattr(self.reasoner.l1_and, "group_mask", None) is not None:
            w = w * self.reasoner.l1_and.group_mask
        return (w >= self.tau_and).float()

    @torch.no_grad()
    def _get_w_and2(self) -> torch.Tensor:
        """W_and2 binarized [r1, r2a]."""
        if hasattr(self.reasoner.l2a_and2, "W"):
            w = self.reasoner.l2a_and2.W.data.clamp(0.0, 1.0)
        else:
            w = torch.sigmoid(self.reasoner.l2a_and2.weight_logit)
        return (w >= self.tau_and2).float()

    @torch.no_grad()
    def _get_w_or1(self) -> torch.Tensor:
        """W_or1 binarized [r1, r2b]."""
        if hasattr(self.reasoner.l2b_or1, "W"):
            w = self.reasoner.l2b_or1.W.data.clamp(0.0, 1.0)
        else:
            w = torch.sigmoid(self.reasoner.l2b_or1.weight_logit)
        return (w >= self.tau_or).float()

    @torch.no_grad()
    def _get_action_weights(self) -> torch.Tensor:
        """ActionHead weights [logic_dim, n_actions]."""
        return self.reasoner.action_head.weight.T

    # ─── Chuyển đổi literal index → tên concept ────────────────────────────

    def _literal_to_name(self, lit_idx: int) -> Tuple[str, bool]:
        n = self.n_concepts
        if lit_idx < n:
            cname = self.concept_names[lit_idx] if lit_idx < len(self.concept_names) else f"c_{lit_idx}"
            return cname, True
        else:
            base_idx = lit_idx - n
            cname = self.concept_names[base_idx] if base_idx < len(self.concept_names) else f"c_{base_idx}"
            return cname, False

    # ─── Trích xuất một L1 rule ─────────────────────────────────────────────

    def _extract_l1_rule(self, w_and1: torch.Tensor, l1_rule_idx: int) -> List[Tuple[str, bool]]:
        active_lits = w_and1[:, l1_rule_idx].nonzero(as_tuple=True)[0].tolist()
        if hasattr(self.reasoner.l1_and, "W"):
            w_col = self.reasoner.l1_and.W.data[:, l1_rule_idx].clamp(0.0, 1.0)
        else:
            w_col = torch.sigmoid(self.reasoner.l1_and.weight_logit[:, l1_rule_idx])
        if getattr(self.reasoner.l1_and, "group_mask", None) is not None:
            w_col = w_col * self.reasoner.l1_and.group_mask[:, l1_rule_idx]
        sorted_lits = sorted(active_lits, key=lambda i: -w_col[i].item())
        sorted_lits = sorted_lits[:self.top_k_literals]
        return [self._literal_to_name(i) for i in sorted_lits]

    # ─── Trích xuất rules cho một action ────────────────────────────────────

    def extract_rules_for_action(self, action_idx: int) -> Dict:
        """Trích xuất các mệnh đề DNF khẳng định (w > tau_act) cho action_idx."""
        w_and1 = self._get_w_and1()
        w_and2 = self._get_w_and2()
        w_or1  = self._get_w_or1()
        W_act  = self._get_action_weights()

        r1 = getattr(self.reasoner, "r1", 256)
        r2a = getattr(self.reasoner, "r2a", 128)
        r2b = getattr(self.reasoner, "r2b", 128)
        act_weights = W_act[:, action_idx]

        clauses = []

        if act_weights.shape[0] == (r1 + r2a + r2b):
            # 512-dim ActionHead: [r_and1 (256); r_and2 (128); r_or1 (128)]
            and1_weights = act_weights[:r1]
            and2_weights = act_weights[r1 : r1 + r2a]
            or1_weights  = act_weights[r1 + r2a :]

            # ── AND1 (L1 Skip Connections) — Primitive Pose Rules
            top_and1_idx = and1_weights.argsort(descending=True)[:self.top_k_l2].tolist()
            for l1_idx in top_and1_idx:
                w_val = and1_weights[l1_idx].item()
                if w_val < self.tau_act:
                    continue
                lits = self._extract_l1_rule(w_and1, l1_idx)
                if lits:
                    clauses.append({
                        "type": "and1",
                        "l1_rule_idx": l1_idx,
                        "l1_components": [{"l1_rule_idx": l1_idx, "literals": lits}],
                        "raw_weight": w_val,
                    })
        else:
            # 256-dim ActionHead (legacy)
            and2_weights = act_weights[:r2a]
            or1_weights  = act_weights[r2a:]

        # ── AND2 (L2a) — chỉ lấy trọng số dương khẳng định
        top_and2_idx = and2_weights.argsort(descending=True)[:self.top_k_l2].tolist()
        for l2_idx in top_and2_idx:
            w_val = and2_weights[l2_idx].item()
            if w_val < self.tau_act:
                continue
            l1_idx_list = w_and2[:, l2_idx].nonzero(as_tuple=True)[0].tolist()[:self.top_k_l1]
            l1_components = []
            for l1_idx in l1_idx_list:
                lits = self._extract_l1_rule(w_and1, l1_idx)
                if lits:
                    l1_components.append({"l1_rule_idx": l1_idx, "literals": lits})
            if l1_components:
                clauses.append({
                    "type": "and2",
                    "l2_rule_idx": l2_idx,
                    "l1_components": l1_components,
                    "raw_weight": w_val,
                })

        # ── OR1 (L2b) — chỉ lấy trọng số dương khẳng định
        top_or1_idx = or1_weights.argsort(descending=True)[:self.top_k_l2].tolist()
        for l2_idx in top_or1_idx:
            w_val = or1_weights[l2_idx].item()
            if w_val < self.tau_act:
                continue
            l1_idx_list = w_or1[:, l2_idx].nonzero(as_tuple=True)[0].tolist()[:self.top_k_l1]
            l1_components = []
            for l1_idx in l1_idx_list:
                lits = self._extract_l1_rule(w_and1, l1_idx)
                if lits:
                    l1_components.append({"l1_rule_idx": l1_idx, "literals": lits})
            if l1_components:
                clauses.append({
                    "type": "or1",
                    "l2_rule_idx": l2_idx,
                    "l1_components": l1_components,
                    "raw_weight": w_val,
                })

        action_name = self.action_names[action_idx] if action_idx < len(self.action_names) else f"action_{action_idx}"
        return {
            "action_name": action_name,
            "action_idx": action_idx,
            "clauses": clauses,
        }


    def extract_all_rules(self) -> List[Dict]:
        """Trích xuất rules cho tất cả actions và lưu cache."""
        rules = [self.extract_rules_for_action(i) for i in range(self.n_actions)]
        self._cached_rules = rules
        return rules

    def get_or_extract_rules(self) -> List[Dict]:
        """Lấy rules đã trích xuất từ cache hoặc trích xuất mới."""
        if not hasattr(self, "_cached_rules") or self._cached_rules is None:
            self._cached_rules = self.extract_all_rules()
        return self._cached_rules

    # ─── Executable DNF Evaluation Engine ─────────────────────────────────────

    @torch.no_grad()
    def evaluate_dnf(
        self,
        z: torch.Tensor,
        rules: Optional[List[Dict]] = None,
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Thực thi chính xác tập luật DNF đã trích xuất (khớp 100% với text/JSON).

        Đánh giá biểu thức boolean của các mệnh đề sau khi đã áp dụng top_k_l2,
        top_k_l1, và top_k_literals, đảm bảo metrics (Coverage, Precision, Fidelity)
        đo lường đúng mô hình DNF thực tế được xuất ra.

        Args:
            z:     Concept logits [B, n_concepts]
            rules: Danh sách rule dicts cho các actions (None = dùng get_or_extract_rules())

        Returns:
            dnf_preds:    [B] predicted class index (-1 nếu không luật nào khớp)
            dnf_scores:   [B, n_actions] activation scores từ các luật DNF
            covered_mask: [B] bool tensor (True nếu có ít nhất 1 luật kích hoạt)
        """
        if rules is None:
            rules = self.get_or_extract_rules()

        device = z.device
        B = z.shape[0]

        # 1. Binarize literals theo ngưỡng z > 0
        hard_pos = (z > 0.0)
        hard_neg = ~hard_pos
        l_bin = torch.cat([hard_pos, hard_neg], dim=-1)  # [B, 2*C] bool tensor

        concept_to_idx = {name: i for i, name in enumerate(self.concept_names)}
        dnf_scores = torch.zeros(B, self.n_actions, device=device)

        # 2. Đánh giá chính xác từng mệnh đề DNF đã trích xuất
        for rule_entry in rules:
            act_idx = rule_entry["action_idx"]
            for clause in rule_entry["clauses"]:
                clause_type = clause["type"]
                w = float(clause["raw_weight"])

                comp_results = []
                for comp in clause["l1_components"]:
                    lits = comp["literals"]
                    if not lits:
                        continue
                    lit_indices = []
                    for cname, is_pos in lits:
                        c_idx = concept_to_idx.get(cname, None)
                        if c_idx is not None and c_idx < self.n_concepts:
                            lit_indices.append(c_idx if is_pos else (c_idx + self.n_concepts))

                    if lit_indices:
                        # L1 rule active iff tất cả literal thành phần thỏa mãn
                        comp_active = l_bin[:, lit_indices].all(dim=-1)  # [B] bool
                        comp_results.append(comp_active)

                if not comp_results:
                    continue

                comp_tensor = torch.stack(comp_results, dim=1)  # [B, num_comps] bool
                if clause_type == "and2":
                    # AND2: Tất cả component L1 rules phải thỏa mãn
                    clause_active = comp_tensor.all(dim=1)  # [B] bool
                elif clause_type == "or1":
                    # OR1: Ít nhất 1 component L1 rule thỏa mãn
                    clause_active = comp_tensor.any(dim=1)  # [B] bool
                else:
                    clause_active = comp_tensor.all(dim=1)

                dnf_scores[:, act_idx] += w * clause_active.float()

        # 3. Mẫu được bao phủ nếu có ít nhất 1 class nhận score > 0
        max_scores, best_class = dnf_scores.max(dim=-1)
        covered_mask = max_scores > 0.0

        dnf_preds = torch.where(covered_mask, best_class, torch.tensor(-1, device=device))

        return dnf_preds, dnf_scores, covered_mask

    # ─── Đánh giá Fidelity, Coverage, Precision theo chuẩn kế hoạch ───────────

    @torch.no_grad()
    def evaluate_fidelity(
        self,
        model,
        data_loader,
        device: str = "cpu",
        n_batches: int = 50,
    ) -> Dict[str, float]:
        """Tính toán: Rule Precision, Rule Coverage, và Model-Rule Fidelity.

        Returns:
            {
                "coverage": float,   # Tỷ lệ mẫu được ít nhất 1 luật DNF bao phủ
                "precision": float,  # Độ chính xác của các dự đoán luật DNF (trên tập covered)
                "fidelity": float,   # Tỷ lệ dự đoán NN trùng khớp với luật DNF executable
                "total_samples": int,
                "covered_samples": int,
            }
        """
        model.eval()
        rules = self.get_or_extract_rules()
        total_samples = 0
        covered_samples = 0
        rule_correct = 0
        model_rule_agree = 0

        for batch_idx, batch in enumerate(data_loader):
            if batch_idx >= n_batches:
                break
            if len(batch) >= 5:
                data, index_t, label, index, _ = batch[:5]
            else:
                data, index_t, label, index = batch[:4]

            data    = data.float().to(device)
            index_t = index_t.float().to(device)
            label   = label.long().to(device)

            output = model(data, index_t=index_t)
            if isinstance(output, dict):
                nn_preds = output["action_logits"].argmax(dim=-1)
                z_final = output["concept_logits"]
            else:
                nn_preds = output.argmax(dim=-1)
                z_final = None

            if z_final is None:
                continue

            # Đánh giá chính xác tập luật DNF đã trích xuất
            dnf_preds, _, covered_mask = self.evaluate_dnf(z_final, rules=rules)

            b_total = len(label)
            total_samples += b_total

            # Coverage
            b_cov = covered_mask.sum().item()
            covered_samples += b_cov

            # Precision (trên các mẫu được bao phủ)
            if b_cov > 0:
                rule_correct += (dnf_preds[covered_mask] == label[covered_mask]).sum().item()

            # Fidelity: NN prediction trùng với DNF prediction
            model_rule_agree += (nn_preds == dnf_preds).sum().item()

        coverage = (covered_samples / total_samples) if total_samples > 0 else 0.0
        precision = (rule_correct / covered_samples) if covered_samples > 0 else 0.0
        fidelity = (model_rule_agree / total_samples) if total_samples > 0 else 0.0

        return {
            "coverage": coverage,
            "precision": precision,
            "fidelity": fidelity,
            "total_samples": total_samples,
            "covered_samples": covered_samples,
        }

    # ─── Format DNF dạng văn bản ─────────────────────────────────────────────

    @staticmethod
    def format_rule_as_dnf(rule: Dict) -> str:
        action_name = rule["action_name"]
        lines = [f"Action '{action_name}' (idx={rule['action_idx']}):"]

        for ci, clause in enumerate(rule["clauses"]):
            clause_type = clause["type"].upper()
            rule_idx = clause.get("l2_rule_idx", clause.get("l1_rule_idx", "?"))
            layer_tag = "L1" if clause_type == "AND1" else "L2"
            w = clause["raw_weight"]
            parts = []
            for comp in clause["l1_components"]:
                lits = comp["literals"]
                lit_strs = [
                    f"{'NOT ' if not pos else ''}{name}"
                    for name, pos in lits
                ]
                parts.append(f"({' ∧ '.join(lit_strs)})")
            connector = " ∨ " if clause_type == "OR1" else " ∧ "
            clause_str = f"  Clause {ci+1} [{clause_type}, {layer_tag}={rule_idx}, w={w:.3f}]:\n    {connector.join(parts)}"
            lines.append(clause_str)


        if not rule["clauses"]:
            lines.append("  (No active affirmative rules extracted)")

        return "\n".join(lines)

    @staticmethod
    def save_rules(rules: List[Dict], output_dir: str):
        """Lưu rules trích xuất ra file JSON và text trong output_dir."""
        import os
        import json
        os.makedirs(output_dir, exist_ok=True)
        json_path = os.path.join(output_dir, "dnf_rules.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(rules, f, ensure_ascii=False, indent=2)

        txt_path = os.path.join(output_dir, "dnf_rules.txt")
        with open(txt_path, "w", encoding="utf-8") as f:
            for rule in rules:
                f.write(RuleExtractor.format_rule_as_dnf(rule) + "\n\n")




## 🧪 Block 7: Kiểm thử nhanh tính toàn vẹn (Quick Smoke Test cho 60CSub)


In [ ]:
# Quick Model Verification Test cho NTU60 Cross-Subject (60 classes)
model_smoke = CLIR_SkateFormer(
    num_classes=60,
    num_concepts=72,
    spatial_concepts=52,
    temporal_concepts=20,
    num_loops=2,
    lambda_sup=1.0,
    lambda_reg=0.005,
)
print("CLIR-SkateFormer (60CSub) initialized successfully!")
print(f"Total Parameters: {sum(p.numel() for p in model_smoke.parameters() if p.requires_grad):,}")

# Test Dummy Forward với 60 action classes
x_dummy = torch.randn(2, 3, 64, 24, 2)
c_targets = torch.randint(0, 2, (2, 72)).float()
a_labels = torch.randint(0, 60, (2,)).long()

# Pure STE Default Check (Epoch 1 setup)
out_default = model_smoke(x_dummy, concept_targets=c_targets, action_labels=a_labels)
print(f"Default Pure STE Loss: {out_default['aux_loss'].item():.4f}, l_reg: {out_default['l_reg'].item():.4f}, beta={model_smoke.reasoner.get_betas().detach().numpy()}")
assert out_default["action_logits"].shape == (2, 60), "Action logits must have shape [2, 60] for 60CSub"

# Phase 1 Fallback Check
model_smoke.set_phase(1)
out1 = model_smoke(x_dummy, concept_targets=c_targets, action_labels=a_labels)
print(f"Phase 1 Fallback Loss: {out1['aux_loss'].item():.4f}, beta={model_smoke.reasoner.get_betas().detach().numpy()}")
assert out1["action_logits"].shape == (2, 60), "Action logits must have shape [2, 60] for 60CSub"

# Phase 2 Check
model_smoke.set_phase(2, temperature=6.0)
out2 = model_smoke(x_dummy, concept_targets=c_targets, action_labels=a_labels)
print(f"Phase 2 Loss: {out2['aux_loss'].item():.4f}, l_reg: {out2['l_reg'].item():.4f}, beta={model_smoke.reasoner.get_betas().detach().numpy()}")
assert out2["action_logits"].shape == (2, 60), "Action logits must have shape [2, 60] for 60CSub"
print("Smoke Test for 60CSub Passed!")



## ⚙️ Block 8: Cấu hình Huấn luyện & Chuẩn bị DataLoaders (NTU60 Cross-Subject)
Mặc định được thiết lập cho **NTU60 Cross-Subject (60CSub)**.

> 💡 **TÙY CHỌN CHẾ ĐỘ HUẤN LUYỆN ĐỂ SO SÁNH**:
> Dưới đây có 2 block cấu hình sẵn sàng. Bạn chỉ cần **comment / uncomment** 1 trong 2 block để so sánh trực tiếp:
> 1. **Block 1**: 🚀 `100% Pure STE Annealing` (Đề xuất cho luận văn - Binarize cứng $\{0, 1\}$, Temperature Annealing $2.0 \rightarrow 10.0$, $\mathcal{L}_{\text{reg}}=0.005$).
> 2. **Block 2**: 🌿 `Pure Soft Baseline` (Để so sánh ablation - Sigmoid liên tục, Không dùng STE, Không binarize, $\mathcal{L}_{\text{reg}}=0.0$).



In [ ]:
# Cấu hình Benchmark (Mặc định NTU60 Cross-Subject / 60CSub)
BENCHMARK = "ntu60_cs"  # @param ["ntu60_cs", "ntu60_cv", "ntu120_csub", "ntu120_cset"]

DATASET_CONFIGS = {
    "ntu60_cs": {
        "num_classes": 60,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CS_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CS_64f.npz",
        "backbone_weights": "/content/ntu60_CS/SkateFormer_j.pt",
        "work_dir": "./work_dir/ntu60_cs/CLIR_SkateFormer_j",
    },
    "ntu60_cv": {
        "num_classes": 60,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CV_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU60_CV_64f.npz",
        "backbone_weights": "/content/ntu60_CV/SkateFormer_j.pt",
        "work_dir": "./work_dir/ntu60_cv/CLIR_SkateFormer_j",
    },
    "ntu120_csub": {
        "num_classes": 120,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSub_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSub_64f.npz",
        "backbone_weights": "/content/ntu120_CSub/SkateFormer_j.pt",
        "work_dir": "./work_dir/ntu120_csub/CLIR_SkateFormer_j",
    },
    "ntu120_cset": {
        "num_classes": 120,
        "train_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSet_64f.npz",
        "test_data": "/content/drive/MyDrive/KLTN/KLTN_Huy-Nhu/NTU120_CSet_64f.npz",
        "backbone_weights": "/content/ntu120_CSet/SkateFormer_j.pt",
        "work_dir": "./work_dir/ntu120_cset/CLIR_SkateFormer_j",
    },
}

selected = DATASET_CONFIGS[BENCHMARK]

# ==============================================================================
# 🎯 LỰA CHỌN CHẾ ĐỘ HUẤN LUYỆN: COMMENT / UNCOMMENT 1 TRONG 2 BLOCK DƯỚI ĐÂY
# ==============================================================================

# --- [BLOCK 1] 🚀 CHẾ ĐỘ 100% PURE STE ANNEALING (MẶC ĐỊNH - ĐỀ XUẤT CHO LUẬN VĂN) ---
# Đặc điểm: Literals nhị phân {0, 1} từ Epoch 1, T anneal 2.0 -> 10.0, L_reg = 0.005
TRAINING_MODE = {
    "mode_name": "Pure_STE_Annealing",
    "literal_mode": "ste",
    "phase1_epochs": 0,             # 0 = Bắt đầu STE ngay từ Epoch 1
    "ste_temperature_start": 2.0,   # T_start = 2.0 chống bão hòa / chết gradient
    "ste_temperature_end": 10.0,    # T_end = 10.0 tiệm cận hàm Heaviside, tối đa Rule Fidelity
    "lambda_reg": 0.005,            # Bật rule sparsity & loop stability
    "beta_max": 0.5,                # Biên độ feedback tinh chỉnh logic K=2
}

# --- [BLOCK 2] 🌿 CHẾ ĐỘ CHỈ SOFT KHÔNG (PURE SOFT BASELINE ĐỂ SO SÁNH ABLATION) ---
# (Để chạy so sánh: Uncomment block dưới và comment block 1 ở trên)
# TRAINING_MODE = {
#     "mode_name": "Pure_Soft",
#     "literal_mode": "soft",
#     "phase1_epochs": 999,           # Chạy Soft toàn bộ quá trình (không bao giờ chuyển STE)
#     "ste_temperature_start": 1.0,   # Không dùng STE
#     "ste_temperature_end": 1.0,
#     "lambda_reg": 0.0,              # Tắt rule sparsity regularization
#     "beta_max": 0.1,                # Giữ beta nhỏ ổn định cho continuous sigmoid
# }

TRAIN_CONFIG = {
    "benchmark": BENCHMARK,
    "num_classes": selected["num_classes"],
    "num_concepts": 72,
    "spatial_concepts": 52,
    "temporal_concepts": 20,
    
    # Đường dẫn dữ liệu
    "train_data": selected["train_data"],
    "test_data": selected["test_data"],
    "backbone_weights": selected["backbone_weights"],
    "work_dir": selected["work_dir"] if TRAINING_MODE.get("mode_name") != "Pure_Soft" else selected["work_dir"] + "_PureSoft",
    
    # Tham số huấn luyện chung
    "batch_size": 32,
    "test_batch_size": 32,
    "num_epoch": 70,
    "base_lr": 1e-4,
    "min_lr": 1e-6,
    "warmup_lr": 1e-6,
    "warm_up_epoch": 2,
    "weight_decay": 0.05,
    "grad_clip": 1.0,
    
    # 🎯 Two-Stage Concept-to-Logic Training (Giải pháp B):
    # Stage 1: Huấn luyện riêng Concept Head (L_c^0) với Reasoner đóng băng
    # Stage 2: Mở khóa Reasoner và ActionHead học End-to-End Logic Reasoning
    "concept_warmup_epochs": 5,   # Khuyên dùng 5 (hoặc 3). Đặt 0 nếu muốn tắt Stage 1
    
    # Học theo nhóm (áp dụng cho cả 2 mode để so sánh công bằng)
    "backbone_lr_scale": 0.1,
    "crl_lr_scale": 1.0,
    "update_cell_lr_scale": 2.0,
    "lambda_sup": 1.0,
    
    # Vòng lặp CLIR
    "num_loops": 2,
    "beta_init": 0.1,
}

# Hợp nhất cấu hình chế độ được chọn
TRAIN_CONFIG.update(TRAINING_MODE)

os.makedirs(TRAIN_CONFIG["work_dir"], exist_ok=True)
print("=" * 75)
print(f"Active Benchmark:     {BENCHMARK.upper()} ({TRAIN_CONFIG['num_classes']} action classes)")
print(f"Active Training Mode: [{TRAINING_MODE.get('mode_name')}]")
if TRAINING_MODE.get('mode_name') == 'Pure_Soft':
    print("                      🌿 PURE SOFT BASELINE (Continuous Sigmoid, No STE, lambda_reg=0.0)")
else:
    print(f"                      🚀 100% PURE STE ANNEALING (T: {TRAIN_CONFIG['ste_temperature_start']} -> {TRAIN_CONFIG['ste_temperature_end']}, lambda_reg={TRAIN_CONFIG['lambda_reg']})")
print(f"Work Directory:       {TRAIN_CONFIG['work_dir']}")
print("=" * 75)



In [ ]:
# Khởi tạo DataLoader nếu file dữ liệu tồn tại
from torch.utils.data import DataLoader

if os.path.exists(TRAIN_CONFIG["train_data"]):
    print(f"Loading {TRAIN_CONFIG['benchmark'].upper()} dataset from disk...")
    train_dataset = Feeder(
        data_path=TRAIN_CONFIG["train_data"],
        concepts_csv="concepts/ntu_rgbd_60_120.csv",
        split="train",
        window_size=64,
        thres=64,
        uniform=True,
        partition=True,
        remove_null_concept=True,
        p_interval=[0.5, 1],
        aug_method="a123489",
        intra_p=0.5,
        inter_p=0.2,
    )
    test_dataset = Feeder(
        data_path=TRAIN_CONFIG["test_data"],
        concepts_csv="concepts/ntu_rgbd_60_120.csv",
        split="test",
        window_size=64,
        thres=64,
        uniform=True,
        partition=True,
        remove_null_concept=True,
        p_interval=[0.95],
    )
    train_loader = DataLoader(train_dataset, batch_size=TRAIN_CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=TRAIN_CONFIG["test_batch_size"], shuffle=False, num_workers=2, pin_memory=True)
    print(f"DataLoaders ready! Train samples: {len(train_dataset)}, Test samples: {len(test_dataset)}")
else:
    print(f"Warning: {TRAIN_CONFIG['train_data']} not found. DataLoaders skipped.")
    train_loader, test_loader = None, None



## 🏋️ Block 9: Vòng lặp Huấn luyện 100% Pure STE & Lưu Checkpoint
- **Pure STE Training (Mặc định)**: Literals nhị phân $\{0, 1\}$ ở forward pass ngay từ Epoch 1, backward nhận gradient qua surrogate sigmoid.
- **Continuous Temperature Annealing**: $T$ tăng tuyến tính từ $2.0 \rightarrow 10.0$ xuyên suốt toàn bộ số epoch (vừa chống chết gradient, vừa tối đa hóa Rule Fidelity).
- **Per-group LRs**: Backbone SkateFormer (scale 0.1), CRL Modules (scale 1.0), UpdateCell (scale 2.0).
- **Scheduler**: 1 chu kỳ Cosine Annealing liên tục kèm 10 epoch warm-up.
- **Fallback 2-Phase**: Nếu cấu hình `phase1_epochs > 0`, hệ thống tự động chạy chế độ 2-Phase (Warm-up Phase 1 $\rightarrow$ STE Phase 2).



In [ ]:
from tqdm import tqdm
import torch.optim as optim
from timm.scheduler.cosine_lr import CosineLRScheduler

class CLIRTrainer:
    def __init__(self, cfg, train_loader, test_loader):
        self.cfg = cfg
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # 1. Khởi tạo mô hình CLIR cho số classes của benchmark
        self.model = CLIR_SkateFormer(
            num_classes=cfg["num_classes"],
            num_concepts=cfg["num_concepts"],
            spatial_concepts=cfg["spatial_concepts"],
            temporal_concepts=cfg["temporal_concepts"],
            backbone_weights=cfg["backbone_weights"] if os.path.exists(cfg["backbone_weights"]) else None,
            num_loops=cfg["num_loops"],
            beta_max=cfg["beta_max"],
            beta_init=cfg["beta_init"],
            lambda_sup=cfg["lambda_sup"],
            lambda_reg=cfg["lambda_reg"],
        ).to(self.device)
        
        self.is_pure_soft = (
            self.cfg.get("literal_mode") == "soft"
            or self.cfg.get("phase1_epochs", 0) >= self.cfg["num_epoch"]
        )

        if self.is_pure_soft:
            self.current_phase = 1
            self.model.set_phase(1)
            self.model.lambda_reg = self.cfg.get("lambda_reg", 0.0)
            if "beta_max" in self.cfg:
                self.model.reasoner.beta_max = self.cfg["beta_max"]
        elif self.cfg.get("phase1_epochs", 0) <= 0:
            self.current_phase = 2
            self.model.set_phase(2, temperature=self.cfg.get("ste_temperature_start", 2.0))
            self.model.lambda_reg = self.cfg.get("lambda_reg", 0.005)
            if "beta_max" in self.cfg:
                self.model.reasoner.beta_max = self.cfg["beta_max"]
        else:
            self.current_phase = 1
            self.model.set_phase(1)

        self.best_acc = 0.0
        self.global_step = 0
        self.phase_step = 0
        self.start_epoch = 0

        # 🎯 Quản lý Two-Stage Concept-to-Logic Training (Giải pháp B):
        # Stage 1: Huấn luyện riêng Concept Head (L_c^0) với Reasoner đóng băng
        # Stage 2: Mở khóa Reasoner và ActionHead học End-to-End Logic Reasoning
        self.current_stage = 1 if self.cfg.get("concept_warmup_epochs", 0) > 0 else 2
        self._set_stage(self.current_stage, verbose=False)

    def _set_stage(self, stage: int, verbose: bool = True):
        self.current_stage = stage
        if stage == 1:
            if verbose:
                print(f"
{'='*70}")
                print(f"[STAGE 1 ACTIVATED] 🎯 CONCEPT WARM-UP (Epoch 1 → {self.cfg.get('concept_warmup_epochs', 0)})")
                print("  • Mục tiêu: Dạy Concept Heads & Backbone nhận diện 72 concepts chuẩn xác.")
                print("  • Đóng băng (Freeze): Tạm khóa CRL Reasoner & ActionHead để chống nhiễu gradient.")
                print("  • Hàm mất mát: Asymmetric BCE Loss trên 72 concepts (L_c^0).")
                print(f"{'='*70}
")
            for p in self.model.reasoner.parameters():
                p.requires_grad = False
            self._init_optimizer()
        else:
            if verbose:
                print(f"
{'='*70}")
                print(f"[STAGE 2 ACTIVATED] 🚀 END-TO-END LOGIC REASONING (Epoch {self.cfg.get('concept_warmup_epochs', 0) + 1} → {self.cfg['num_epoch']})")
                print("  • Mục tiêu: Mở khóa CRL Reasoner & ActionHead để học các luật DNF và phân loại hành động.")
                print("  • Dữ liệu vào: Concept logits z^0 đã được định hình sạch sẽ từ Stage 1.")
                print("  • Hàm mất mát: Action Cross-Entropy + Auxiliary Concept Loss.")
                print(f"{'='*70}
")
            for p in self.model.reasoner.parameters():
                p.requires_grad = True
            self._init_optimizer()

    def _init_optimizer(self):
        base_lr = self.cfg["base_lr"]

        if self.current_stage == 1:
            # Stage 1: Chỉ tối ưu hóa Backbone và Concept Heads
            bb_scale = self.cfg.get("backbone_lr_scale", 0.1)
            backbone_params = list(self.model.backbone.parameters()) +                               list(self.model.spatial_pool_score.parameters()) +                               list(self.model.temporal_pool_score.parameters())
            head_params = list(self.model.spatial_head.parameters()) +                           list(self.model.temporal_head.parameters())
            params = [
                {"params": backbone_params, "lr": base_lr * bb_scale, "name": "backbone"},
                {"params": head_params, "lr": base_lr, "name": "concept_heads"},
            ]
            self.optimizer = optim.AdamW(params, lr=base_lr, weight_decay=self.cfg["weight_decay"])

            if self.train_loader is not None:
                n_iter = len(self.train_loader)
                warmup_epochs = max(1, self.cfg.get("concept_warmup_epochs", 1))
                total_steps = warmup_epochs * n_iter
                self.scheduler = CosineLRScheduler(
                    self.optimizer, t_initial=total_steps, lr_min=self.cfg["min_lr"],
                    warmup_t=0, cycle_limit=1, t_in_epochs=False,
                )
            return

        # Stage 2: Mở toàn bộ mô hình
        is_staged_p1 = (
            self.current_phase == 1
            and not getattr(self, "is_pure_soft", False)
            and 0 < self.cfg.get("phase1_epochs", 0) < self.cfg["num_epoch"]
        )
        if is_staged_p1:
            params = [{"params": self.model.parameters(), "lr": base_lr}]
        else:
            bb_scale = self.cfg["backbone_lr_scale"]
            crl_scale = self.cfg["crl_lr_scale"]
            uc_scale = self.cfg["update_cell_lr_scale"]

            backbone_params = list(self.model.backbone.parameters()) +                               list(self.model.spatial_pool_score.parameters()) +                               list(self.model.temporal_pool_score.parameters()) +                               list(self.model.spatial_head.parameters()) +                               list(self.model.temporal_head.parameters())
            crl_params = list(self.model.reasoner.crl.parameters())
            if getattr(self.model, "bypass_head", None) is not None:
                crl_params += list(self.model.bypass_head.parameters())
            uc_params = list(self.model.reasoner.update_cell.parameters()) + [self.model.reasoner.raw_beta]

            params = [
                {"params": backbone_params, "lr": base_lr * bb_scale, "name": "backbone"},
                {"params": crl_params, "lr": base_lr * crl_scale, "name": "crl"},
                {"params": uc_params, "lr": base_lr * uc_scale, "name": "update_cell"},
            ]

        self.optimizer = optim.AdamW(params, lr=base_lr, weight_decay=self.cfg["weight_decay"])

        if self.train_loader is not None:
            n_iter = len(self.train_loader)
            stage1_epochs = self.cfg.get("concept_warmup_epochs", 0)
            remaining_epochs = max(1, self.cfg["num_epoch"] - stage1_epochs)
            total_steps = remaining_epochs * n_iter
            warmup_steps = min(int(self.cfg.get("warm_up_epoch", 2) * n_iter), int(total_steps * 0.1))
            self.scheduler = CosineLRScheduler(
                self.optimizer, t_initial=total_steps, lr_min=self.cfg["min_lr"],
                warmup_lr_init=self.cfg["warmup_lr"], warmup_t=warmup_steps, cycle_limit=1, t_in_epochs=False,
            )

    def check_stage_and_phase(self, epoch):
        # 1. Kiểm tra chuyển đổi Stage (Stage 1 -> Stage 2)
        warmup_ep = self.cfg.get("concept_warmup_epochs", 0)
        target_stage = 1 if (warmup_ep > 0 and epoch < warmup_ep) else 2
        if target_stage != self.current_stage:
            self._set_stage(target_stage, verbose=True)
            self.phase_step = 0

        # 2. Kiểm tra nhiệt độ STE / Phase switch khi ở Stage 2
        if self.current_stage == 2:
            self.check_phase_switch(epoch)

    def check_phase_switch(self, epoch):
        if getattr(self, "is_pure_soft", False):
            return

        stage1_ep = self.cfg.get("concept_warmup_epochs", 0)
        effective_epoch = max(0, epoch - stage1_ep)
        effective_total = max(1, self.cfg["num_epoch"] - stage1_ep - 1)

        if self.cfg.get("phase1_epochs", 0) <= 0:
            # Pure STE continuous annealing từ 2.0 → 10.0 trong Stage 2
            progress = min(1.0, max(0.0, effective_epoch / effective_total))
            temp = self.cfg["ste_temperature_start"] + progress * (self.cfg["ste_temperature_end"] - self.cfg["ste_temperature_start"])
            self.model.reasoner.temperature = temp
            return

        new_phase = 1 if epoch < self.cfg["phase1_epochs"] else 2
        if new_phase != self.current_phase:
            print(f"\n{'='*60}\n[PHASE SWITCH] Switching from Phase {self.current_phase} to Phase {new_phase} at Epoch {epoch+1}\n{'='*60}")
            self.current_phase = new_phase
            self.model.set_phase(new_phase)
            self._init_optimizer()
            self.phase_step = 0

        if self.current_phase == 2:
            rem = max(1, self.cfg["num_epoch"] - self.cfg["phase1_epochs"])
            progress = (epoch - self.cfg["phase1_epochs"]) / rem
            temp = self.cfg["ste_temperature_start"] + progress * (self.cfg["ste_temperature_end"] - self.cfg["ste_temperature_start"])
            self.model.reasoner.temperature = temp

    def train_epoch(self, epoch):
        self.model.train()
        self.check_stage_and_phase(epoch)

        is_stage1 = (self.current_stage == 1)
        total_loss, correct, total = 0.0, 0, 0

        if is_stage1:
            mode_tag = f"Stage 1: Concept-Warmup ({epoch+1}/{self.cfg.get('concept_warmup_epochs', 0)})"
        elif getattr(self, "is_pure_soft", False):
            mode_tag = "Stage 2: Pure-Soft"
        elif self.current_phase == 2:
            mode_tag = f"Stage 2: Pure-STE T={self.model.reasoner.temperature:.2f}"
        else:
            mode_tag = f"Stage 2: Phase-{self.current_phase}"

        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch+1}/{self.cfg['num_epoch']} ({mode_tag})")
        for batch in pbar:
            if len(batch) == 5:
                data, index_t, label, _, c_targets = batch
            else:
                data, index_t, label, _ = batch[:4]
                c_targets = None

            data = data.float().to(self.device)
            index_t = index_t.float().to(self.device)
            label = label.long().to(self.device)
            c_targets = c_targets.float().to(self.device) if c_targets is not None else None

            # Step scheduler
            if self.current_stage == 1:
                self.scheduler.step_update(self.global_step)
            else:
                self.scheduler.step_update(self.phase_step)
                self.phase_step += 1
            self.global_step += 1

            self.optimizer.zero_grad()

            if is_stage1:
                # Stage 1: Huấn luyện riêng Concept Head bằng Asymmetric BCE Loss
                out = self.model(data, index_t=index_t, concept_targets=c_targets)
                loss = out.get("l_c_0", None)
                if loss is None:
                    loss = self.model._asymmetric_bce(out["initial_concept_logits"], c_targets)

                loss.backward()
                if self.cfg["grad_clip"] > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg["grad_clip"])
                self.optimizer.step()

                total_loss += loss.item()
                c_pred = (out["initial_concept_logits"] > 0.0).float()
                c_acc = (c_pred == c_targets).float().mean().item() if c_targets is not None else 0.0
                total += 1
                correct += c_acc
                pbar.set_postfix({"c_loss": f"{loss.item():.4f}", "c_acc": f"{correct/total*100:.1f}%"})
            else:
                # Stage 2: Huấn luyện End-to-End đầy đủ (Action Cross-Entropy + Logic Aux Loss)
                out = self.model(data, index_t=index_t, concept_targets=c_targets, action_labels=label)
                loss = torch.nn.functional.cross_entropy(out["action_logits"], label)
                if "aux_loss" in out:
                    loss += out["aux_loss"]

                loss.backward()
                if self.cfg["grad_clip"] > 0:
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg["grad_clip"])
                self.optimizer.step()
                if hasattr(self.model, "clip_weights"):
                    self.model.clip_weights()

                total_loss += loss.item()
                pred = out["action_logits"].argmax(dim=-1)
                correct += (pred == label).sum().item()
                total += label.size(0)
                pbar.set_postfix({"loss": f"{loss.item():.4f}", "acc": f"{correct/total*100:.2f}%"})

        return total_loss / len(self.train_loader), correct / total

    @torch.no_grad()
    def eval_epoch(self):
        self.model.eval()
        correct, total = 0, 0
        for batch in tqdm(self.test_loader, desc="Evaluation"):
            if len(batch) >= 4:
                data, index_t, label = batch[0].float().to(self.device), batch[1].float().to(self.device), batch[2].long().to(self.device)
            out = self.model(data, index_t=index_t)
            logits = out["action_logits"] if isinstance(out, dict) else out
            pred = logits.argmax(dim=-1)
            correct += (pred == label).sum().item()
            total += label.size(0)
        acc = correct / total if total > 0 else 0.0
        return acc

    def _build_checkpoint(self, epoch):
        # Gói đầy đủ state để resume, khớp optimizer/phase hiện tại (Phase 1 hoặc 2)
        return {
            "epoch": epoch + 1,
            "global_step": self.global_step,
            "phase_step": self.phase_step,
            "current_stage": self.current_stage,
            "current_phase": self.current_phase,
            "best_acc": self.best_acc,
            "model": self.model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
        }

    def load_checkpoint(self, checkpoint_path, resume_training=True):
        """Nạp checkpoint vào mô hình và khôi phục trạng thái huấn luyện nếu cần."""
        if not os.path.exists(checkpoint_path):
            print(f"[CLIRTrainer] ⚠️ Không tìm thấy checkpoint tại: {checkpoint_path}")
            return False

        print(f"[CLIRTrainer] 🔄 Đang nạp checkpoint từ: {checkpoint_path}")
        ckpt = torch.load(checkpoint_path, map_location=self.device)

        # Trích xuất model state_dict (hỗ trợ checkpoint đầy đủ hoặc pure state_dict)
        state_dict = ckpt.get("model", ckpt) if isinstance(ckpt, dict) else ckpt
        clean_state = {k.replace("module.", ""): v for k, v in state_dict.items()}
        missing, unexpected = self.model.load_state_dict(clean_state, strict=False)
        print(f"[CLIRTrainer] ✓ Đã nạp weights thành công. Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")

        if resume_training and isinstance(ckpt, dict) and "epoch" in ckpt:
            self.start_epoch = ckpt.get("epoch", 0)
            self.global_step = ckpt.get("global_step", 0)
            self.phase_step = ckpt.get("phase_step", 0)
            self.best_acc = ckpt.get("best_acc", 0.0)

            # Khôi phục đúng Stage theo start_epoch
            warmup_ep = self.cfg.get("concept_warmup_epochs", 0)
            target_stage = 1 if (warmup_ep > 0 and self.start_epoch < warmup_ep) else 2
            self._set_stage(target_stage, verbose=True)

            if self.current_stage == 2:
                self.current_phase = ckpt.get("current_phase", self.current_phase)
                if hasattr(self.model, "set_phase"):
                    self.model.set_phase(self.current_phase)
                self.check_phase_switch(self.start_epoch)

            # Nạp trạng thái optimizer
            if "optimizer" in ckpt and ckpt["optimizer"] is not None:
                try:
                    self.optimizer.load_state_dict(ckpt["optimizer"])
                    print("[CLIRTrainer] ✓ Đã khôi phục trạng thái Optimizer.")
                except Exception as e:
                    print(f"[CLIRTrainer] ⚠️ Không thể khôi phục optimizer state ({e}). Sẽ dùng optimizer mới khởi tạo.")

            print(f"★ [CLIRTrainer] Khôi phục phiên huấn luyện: Bắt đầu từ Epoch {self.start_epoch + 1}/{self.cfg['num_epoch']} (Stage {self.current_stage}, Phase {self.current_phase}, Best Acc: {self.best_acc*100:.2f}%)")
        else:
            print("★ [CLIRTrainer] Đã nạp weights (không khôi phục optimizer/epoch metadata).")

        return True

    def run(self):
        start_ep = getattr(self, "start_epoch", 0)
        total_ep = self.cfg["num_epoch"]
        if start_ep >= total_ep:
            print(f"[CLIRTrainer] Quá trình huấn luyện đã hoàn tất: start_epoch ({start_ep}) >= num_epoch ({total_ep}).")
            return

        print(f"Starting training from Epoch {start_ep + 1} to {total_ep} on {self.cfg['benchmark'].upper()}...")
        for epoch in range(start_ep, total_ep):
            train_loss, train_acc = self.train_epoch(epoch)

            # Lưu checkpoint mới nhất mỗi epoch (đầy đủ optimizer, khớp stage & phase hiện tại)
            latest_path = os.path.join(self.cfg["work_dir"], "checkpoint_latest.pt")
            torch.save(self._build_checkpoint(epoch), latest_path)

            if (epoch + 1) % 5 == 0 or (epoch + 1) == self.cfg["num_epoch"]:
                val_acc = self.eval_epoch()
                stage_tag = "Stage 1 (Concept-Warmup)" if self.current_stage == 1 else "Stage 2 (Logic)"
                metric_name = "Mean Concept Acc" if self.current_stage == 1 else "Train Acc"
                print(f"[Epoch {epoch+1} - {stage_tag}] Train Loss: {train_loss:.4f}, {metric_name}: {train_acc*100:.2f}%, Val Acc: {val_acc*100:.2f}%")

                # Save best checkpoint (chỉ lưu best ở Stage 2 khi đã có suy luận action)
                if (self.current_stage == 2 or self.cfg.get("concept_warmup_epochs", 0) <= 0) and val_acc > self.best_acc:
                    self.best_acc = val_acc
                    save_path = os.path.join(self.cfg["work_dir"], "best_model.pt")
                    torch.save(self.model.state_dict(), save_path)
                    best_ckpt_path = os.path.join(self.cfg["work_dir"], "best_checkpoint.pt")
                    torch.save(self._build_checkpoint(epoch), best_ckpt_path)
                    print(f"★ Saved new best model to {save_path} (Acc: {self.best_acc*100:.2f}%)")

print("CLIRTrainer defined successfully!")



In [ ]:
# Chạy huấn luyện (Bỏ comment để bắt đầu huấn luyện nếu đã có dữ liệu)
# Tùy chọn Resume Checkpoint:
# - Nếu muốn tiếp tục huấn luyện từ lần chạy trước khi Colab bị ngắt, trỏ đến checkpoint_latest.pt
# - Nếu muốn huấn luyện mới hoàn toàn từ Epoch 1, đặt RESUME_CHECKPOINT = None
# RESUME_CHECKPOINT = os.path.join(TRAIN_CONFIG["work_dir"], "checkpoint_latest.pt")
# RESUME_CHECKPOINT = None

# if train_loader is not None and test_loader is not None:
#     trainer = CLIRTrainer(TRAIN_CONFIG, train_loader, test_loader)
#     if RESUME_CHECKPOINT and os.path.exists(RESUME_CHECKPOINT):
#         trainer.load_checkpoint(RESUME_CHECKPOINT, resume_training=True)
#     trainer.run()



## 📜 Block 10: Trích xuất Luật Logic DNF & Đánh giá Tính giải thích (Rule Extraction)
Sau khi có checkpoint đã huấn luyện (hoặc dùng `model_smoke` để thử nghiệm), trích xuất các mệnh đề DNF tường minh cho 60 hành động NTU60 và đánh giá **Rule Coverage**, **Rule Precision**, **Model-Rule Fidelity**.



In [ ]:
# Đọc danh sách tên concept và action (khớp với số lớp của benchmark)
concept_names, action_names = [], []
with open("concepts/ntu_rgbd_60_120.csv", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]
    concept_names = [h.strip() for h in lines[0].split(",")[1:]]
    action_names = [l.split(",")[0].strip() for l in lines[1:]]

# Lọc đúng 60 actions cho NTU60 Cross-Subject
action_names = action_names[:TRAIN_CONFIG["num_classes"]]
print(f"Loaded {len(concept_names)} concepts and {len(action_names)} actions ({TRAIN_CONFIG['benchmark'].upper()}).")

# Khởi tạo mô hình & nạp trọng số đã huấn luyện (nếu có best_model.pt hoặc best_checkpoint.pt)
best_model_path = os.path.join(TRAIN_CONFIG["work_dir"], "best_model.pt")
best_ckpt_path = os.path.join(TRAIN_CONFIG["work_dir"], "best_checkpoint.pt")

if os.path.exists(best_model_path) or os.path.exists(best_ckpt_path):
    load_path = best_model_path if os.path.exists(best_model_path) else best_ckpt_path
    print(f"★ Nạp mô hình đã huấn luyện từ: {load_path}")
    eval_model = CLIR_SkateFormer(
        num_classes=TRAIN_CONFIG["num_classes"],
        num_concepts=TRAIN_CONFIG["num_concepts"],
        spatial_concepts=TRAIN_CONFIG["spatial_concepts"],
        temporal_concepts=TRAIN_CONFIG["temporal_concepts"],
        num_loops=TRAIN_CONFIG["num_loops"],
        beta_max=TRAIN_CONFIG["beta_max"],
        beta_init=TRAIN_CONFIG["beta_init"],
    ).to("cuda" if torch.cuda.is_available() else "cpu")
    
    ckpt_data = torch.load(load_path, map_location="cpu")
    w = ckpt_data.get("model", ckpt_data) if isinstance(ckpt_data, dict) else ckpt_data
    eval_model.load_state_dict({k.replace("module.", ""): v for k, v in w.items()}, strict=False)
    eval_model.eval()
    model_for_extract = eval_model.reasoner.crl
    model_for_eval = eval_model
    print("★ Đã nạp thành công trọng số thực tế để trích xuất luật DNF!")
else:
    print("ℹ Chưa tìm thấy checkpoint huấn luyện thực tế. Sử dụng model_smoke để minh họa luồng trích xuất luật.")
    model_for_extract = model_smoke.reasoner.crl
    model_for_eval = model_smoke

# Khởi tạo RuleExtractor
extractor = RuleExtractor(
    reasoner=model_for_extract,
    concept_names=concept_names,
    action_names=action_names,
    top_k_l2=3,
    top_k_l1=3,
    top_k_literals=4,
)

# Trích xuất luật DNF cho tất cả actions
os.makedirs("rules_output", exist_ok=True)
all_rules = extractor.extract_all_rules()

print(f"Successfully extracted DNF rules for all {len(all_rules)} actions!\n")
print("--- Ví dụ 3 luật DNF trích xuất mẫu ---")
for act_idx in [0, 1, 2]:
    text_rule = RuleExtractor.format_rule_as_dnf(all_rules[act_idx])
    print(text_rule)
    print("-" * 50)

# Lưu vào file
RuleExtractor.save_rules(all_rules, "rules_output")
print("Saved rules to: rules_output/dnf_rules.json and rules_output/dnf_rules.txt")



In [ ]:
# Đánh giá Fidelity, Coverage, Precision trên tập kiểm tra thực tế
if test_loader is not None:
    print("Evaluating DNF rule metrics on test set...")
    metrics = extractor.evaluate_fidelity(model_for_eval, test_loader, device="cuda" if torch.cuda.is_available() else "cpu", n_batches=30)
    print("\n" + "="*50)
    print("🎯 DNF RULE EXPLAINABILITY EVALUATION")
    print("="*50)
    print(f"  • Rule Coverage:       {metrics['coverage']*100:.2f}%  (Tỷ lệ mẫu có ít nhất 1 luật kích hoạt)")
    print(f"  • Rule Precision:      {metrics['precision']*100:.2f}%  (Độ chính xác khi luật DNF đưa ra dự đoán)")
    print(f"  • Model-Rule Fidelity: {metrics['fidelity']*100:.2f}%  (Độ tương đồng giữa luật và mạng nơ-ron)")
    print("="*50)
else:
    print("Test loader not active. DNF evaluation on dummy concept logits:")
    z_eval = torch.randn(8, 72)
    preds, scores, cov = extractor.evaluate_dnf(z_eval)
    print(f"Evaluated on 8 samples. Covered: {cov.sum().item()}/8")



## 💻 Block 11: Huấn luyện & Trích xuất qua Dòng lệnh CLI (NTU60 Cross-Subject)
Nếu bạn muốn huấn luyện theo script chuẩn hóa qua terminal hoặc cell Colab:



In [ ]:
# 1. Huấn luyện qua dòng lệnh trên NTU RGB+D 60 Cross-Subject (60CSub)
!python main.py --config config/train/ntu60_cs/CLIR_SkateFormer_j.yaml

# 2. Tiếp tục huấn luyện (Resume) từ checkpoint nếu bị gián đoạn:
# !python main.py --config config/train/ntu60_cs/CLIR_SkateFormer_j.yaml \
#     --weights work_dir/ntu60_cs/CLIR_SkateFormer_j/checkpoint_latest.pt \
#     --start-epoch 35

# 3. Chỉ đánh giá (Test/Eval) checkpoint tốt nhất:
# !python main.py --config config/train/ntu60_cs/CLIR_SkateFormer_j.yaml \
#     --weights work_dir/ntu60_cs/CLIR_SkateFormer_j/best_model.pt \
#     --phase test

# 4. Huấn luyện các biến thể Ablation trên NTU60 CS
# Ablation A2: True No-Loop (beta=0, num_loops=0)
# !python main.py --config config/train/ntu60_cs/CLIR_SkateFormer_j_A2_NoLoop.yaml

# Ablation A5: Random Logic Feedback
# !python main.py --config config/train/ntu60_cs/CLIR_SkateFormer_j_A5_RandomFeedback.yaml

# Ablation A6: CLIR-Bypass (Feature bypass branch)
# !python main.py --config config/train/ntu60_cs/CLIR_SkateFormer_j_A6_Bypass.yaml

# 5. Trích xuất luật DNF sau huấn luyện từ file checkpoint
# !python scripts/extract_rules.py \
#     --checkpoint work_dir/ntu60_cs/CLIR_SkateFormer_j/best_model.pt \
#     --config config/train/ntu60_cs/CLIR_SkateFormer_j.yaml \
#     --output-dir work_dir/rules_ntu60_cs \
#     --eval-fidelity

